# iGEM Pooled NLP - Notebook 1: Embedding & Core Pooled Artifacts

**Restart-safe, artifact-driven.** Run-once expensive work: environment, local model cache, persistent artifact store, control panel, data load, cleaning, Qwen3 embedding, embedding-quality checks, outlier handling, and pooling the five questions. Saves every hand-off artifact (local + optional persistent mirror; files over the size cap stay local) and writes `embed_manifest.json`.

Run this once. When embeddings are good, go to Notebook 2 (clustering) then Notebook 3 (visuals).

> **Pipeline order (corrected).** Outlier screening and removal now run **before** cleaning and embedding. In the previous version they ran *after* the embedding stage, and pooling consumed the embeddings computed from the unfiltered text — so the screening printed a full diagnostic but changed nothing downstream. `minimal_clean` also kept only Latin and CJK characters, which silently deleted Cyrillic, Devanagari, Hangul, Greek and Arabic responses and reduced Japanese to a kanji fragment; it is now Unicode-category based, so the multilingual embedder sees every script.


## Stage 1 — Environment

Recommended Colab runtime: an L4 GPU with High-RAM (Runtime, Change runtime type). Installs the libraries and sets the determinism environment variables that must exist before CUDA starts. Parametric UMAP and TensorFlow are not used: this pipeline relies on standard, seeded UMAP, which is reproducible. Run once per session.

In [ ]:
# @title
# ==============================================================================
# STAGE 1  -  ENVIRONMENT
# Installs the libraries the pipeline needs, sets the determinism environment
# variables that must exist before CUDA initialises, and prints a version
# fingerprint for the methods section. Run once per session.
# Default embedding is Qwen3-Embedding-4B (needs recent transformers /
# sentence-transformers). Parametric UMAP and TensorFlow are intentionally NOT
# installed: this rebuild uses standard, seeded UMAP (bit-for-bit reproducible).
# ==============================================================================
import os

# Determinism vars set before torch / CUDA touch the GPU.
os.environ["PYTHONHASHSEED"]        = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS matmuls
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
def _pip(pkgs):
    subprocess.run(f"pip install -q {pkgs}", shell=True, check=False)

# One install pass. Pinned where a known-good version matters (kaleido 0.1.x
# exports static Plotly images headlessly on Colab).
_pip("'sentence-transformers>=3.3.0' 'transformers>=4.51.0' umap-learn scikit-learn scipy")
_pip("torch bitsandbytes accelerate")
_pip("lingua-language-detector contractions 'spacy>=3.8'")
_pip("plotly 'kaleido==0.1.0.post1' matplotlib seaborn colorcet gdown")
subprocess.run("python -m spacy download en_core_web_sm -q", shell=True, check=False)
_pip("nltk")

import torch
from importlib.metadata import version, PackageNotFoundError
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print("=" * 64)
print("  STAGE 1  ENVIRONMENT")
print("=" * 64)
print(f"  Device : {gpu}")
print("  Versions (record these for the methods section / requirements.txt):")
for p in ["torch", "transformers", "sentence-transformers", "umap-learn",
          "scikit-learn", "scipy", "numpy", "plotly", "kaleido"]:
    try:    print(f"    {p:22s} {version(p)}")
    except PackageNotFoundError: print(f"    {p:22s} (not found)")
print("=" * 64)


## Stage 2 — Drive and model cache

This pipeline does not need Google Drive. Model weights and results both live on Colab local disk, so by default Drive is not mounted at all (no connect step, no auth popup, no Drive errors), and nothing fills your Drive. The survey data is fetched by its public share link, which also needs no mount. Set RESULTS_ON_DRIVE=True only if you want results to persist across sessions, in which case Drive is mounted with a safe fallback to local disk.

In [ ]:
# @title
# ==============================================================================
# STAGE 2  -  STORAGE, MODEL CACHE & PERSISTENT ARTIFACTS  (rewritten)
# Two concerns are kept strictly apart:
#   * MODEL CACHE  (HF_HOME)       -> ALWAYS local disk. The embedder / LLM weights
#                                     are many GB and must never be written to Drive.
#   * ARTIFACT STORE (CHECKPOINT)  -> local disk by default, with an OPTIONAL
#                                     persistent MIRROR (Google Drive on Colab, or
#                                     any folder you pick) for files <= the size cap.
# This implements the plan exactly: every artifact is written LOCALLY and also
# MIRRORED to a persistent place, EXCEPT a single file larger than
# ARTIFACT_SIZE_CAP_MB, which stays LOCAL-ONLY (so a big embedding/UMAP matrix
# never tries to fill Drive). Notebook 2 reads the SAME store: local first, then
# the mirror, so you never recompute embeddings just because the runtime reset.
# ==============================================================================
import os, shutil

# ------------------------------ EDIT HERE -------------------------------------
USE_PERSISTENT_MIRROR = True     # True = also copy small artifacts to a persistent folder
PERSIST_ROOT          = None     # None = auto (mount Google Drive on Colab).
                                 # Or set a path, e.g. "/content/drive/MyDrive"
                                 # or a local persistent disk / mounted folder.
ARTIFACT_SIZE_CAP_MB  = 100      # a single file larger than this is saved LOCAL ONLY
# ----------------------------- END EDIT ---------------------------------------

ON_COLAB   = os.path.isdir("/content")
LOCAL_BASE = "/content" if ON_COLAB else os.path.abspath(".")

# Model weights -> ALWAYS local disk.
os.environ["HF_HOME"] = os.path.join(LOCAL_BASE, "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Local artifact root (fast; may be ephemeral on Colab).
CHECKPOINT_DIR = os.path.join(LOCAL_BASE, "iGEM_Pipeline_Checkpoints_Pooled")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Optional persistent mirror (Drive on Colab, or a folder you choose).
MIRROR_ROOT = None
if USE_PERSISTENT_MIRROR:
    _root = PERSIST_ROOT
    if _root is None and ON_COLAB:
        try:
            from google.colab import drive
            try:
                drive.mount("/content/drive", force_remount=False)
            except ValueError:
                pass  # already mounted
            _root = "/content/drive/MyDrive"
        except Exception as e:
            print(f"  Drive mount unavailable ({type(e).__name__}); mirror OFF, local only.")
            _root = None
    if _root:
        MIRROR_ROOT = os.path.join(_root, "iGEM_Pipeline_Checkpoints_Pooled")
        try:
            os.makedirs(MIRROR_ROOT, exist_ok=True)
        except Exception as e:
            print(f"  Could not create mirror dir ({type(e).__name__}); local only.")
            MIRROR_ROOT = None

# Notebook identity (stamped into every manifest).
NOTEBOOK_NAME    = "Notebook 1 - Embedding + Pooled Artifacts"
NOTEBOOK_VERSION = "2.0"

print("=" * 64)
print("  STAGE 2  STORAGE & MODEL CACHE")
print("=" * 64)
print(f"  notebook        : {NOTEBOOK_NAME}  (v{NOTEBOOK_VERSION})")
print(f"  model cache     : {os.environ['HF_HOME']}  (LOCAL, always)")
print(f"  artifact local  : {CHECKPOINT_DIR}")
print(f"  artifact mirror : {MIRROR_ROOT or 'OFF (local only)'}")
print(f"  size rule       : files > {ARTIFACT_SIZE_CAP_MB} MB are LOCAL-ONLY (never mirrored)")
print("=" * 64)


## Stage 3 — Control Panel

The single source of truth. Every tunable lives here and is frozen into one immutable `CFG`. `set_all_seeds` fixes Python, NumPy and PyTorch randomness; `CFG.fingerprint` builds the per-stage cache key. Change a value, re-run, and only the affected stages recompute.

In [ ]:
# @title
# ==============================================================================
# STAGE 3  -  CONTROL PANEL  (the single source of truth)
# Every tunable lives here and nowhere else (Golden Rule 3). Edit a value in the
# EDIT block, re-run this cell and the cells below; the content-hash cache
# (Stage 4) recomputes only the stages a change actually affects. The values are
# then frozen into one immutable CFG object that also carries:
#   set_all_seeds()  - fixes Python, NumPy and PyTorch randomness (Golden Rule 1)
#   CFG.fingerprint() - the per-stage cache key (Golden Rule 2)
# ==============================================================================
from dataclasses import dataclass, field, asdict
import os, json, hashlib, random
import numpy as np

# ----------------------------- EDIT HERE --------------------------------------
SEED            = 42
CACHE_VERSION   = "v1"              # bump to "v2","v3"... to force EVERY cached stage to recompute

# Data
DATA_URL        = "https://drive.google.com/uc?id=YOUR_SURVEY_DATA_FILE_ID"
TEXT_COLS       = ("Connection and belonging",
                   "Reflect on how your team worked together",
                   "Beyond iGEM",
                   "Supporting and limiting conditions",
                   "One word or phrase that captures what iGEM meant to you")
SHORT_LABELS    = ("Connection", "Team Work", "Beyond iGEM", "Conditions", "One Word")

# Embedding   (Qwen3-Embedding-4B by default, 2560-d; switch to "bge-m3" to compare)
EMBED_BACKEND   = "qwen3-8b"        # "qwen3-8b"=Qwen3-Embedding-8B (4096-d, for L4 High-RAM) | "qwen3"=4B (2560-d) | "bge-m3"=1024-d
EMBED_BATCH     = 16

# UMAP  (seeded, standard; no Parametric UMAP)
UMAP_NEIGHBORS  = 50
UMAP_MIN_DIST   = 0.0               # tight geometry for clustering
CLUSTER_DIM     = 5                 # dimensionality of the clustering space

# Ward clustering
K_MODE          = "auto"            # "auto" = data-driven over [K_MIN, K_MAX]; "fixed" = K_FIXED
K_FIXED         = 16
K_MIN           = 4
K_MAX           = 30

# Sub-theme drill-down + validation
SUBTHEME_MIN    = 30                # split any macro theme with at least this many responses
SUBTHEME_TARGET = 50                # aim ~this many responses per sub-theme (LOWER = MORE sub-themes)
SUBTHEME_MAX_K  = 12                # cap sub-themes per macro theme
N_SEEDS         = 10                # multi-seed stability runs
N_BOOTSTRAP     = 100               # bootstrap resamples for Jaccard stability
TRUST_K         = 15                # neighbours for trustworthiness / continuity

# Language-model theme labelling
LLM_ENABLE      = True
LLM_MODEL       = "Qwen/Qwen2.5-14B-Instruct"   # A100: "Qwen/Qwen2.5-32B-Instruct"
LLM_N_QUOTES    = 7
LLM_FORCE_REGEN = False             # True forces re-labelling even if cached

# Team awards (structured metadata, never embedded)
AWARD_YEAR      = 2025
# ------------------------------ END EDIT --------------------------------------

# Embedding registry: name + output dimension for each backend.
_EMBED_REGISTRY = {
    "qwen3":      ("Qwen/Qwen3-Embedding-4B",   2560),  # default: strong + multilingual, fits an L4
    "qwen3-8b":   ("Qwen/Qwen3-Embedding-8B",   4096),  # A100: highest accuracy
    "qwen3-0.6b": ("Qwen/Qwen3-Embedding-0.6B", 1024),  # light alternative
    "bge-m3":     ("BAAI/bge-m3",               1024),  # previous model, kept for comparison
}
# Qwen3-Embedding is instruction-aware: a short task description sharpens the
# geometry for our use (thematic clustering). BGE-M3 ignores this (encoded plain).
EMBED_INSTRUCTION = ("Represent this iGEM participant's survey response BY ITS TOPIC for "
                     "thematic clustering, independent of sentiment or how positive or "
                     "negative it is:")
_EMBED_MODEL, _EMBED_DIM = _EMBED_REGISTRY[EMBED_BACKEND]

# Which settings affect which stage. A stage's cache key is built only from the
# settings above it in the dependency chain, so changing K re-runs Ward but not
# the embedding, and changing UMAP re-runs UMAP, Ward and everything downstream.
_RELEVANT = {
    "data":    ["data_url"],
    "embed":   ["seed", "embed_model", "embed_dim", "embed_batch"],
    "umap":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim"],
    "ward":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max"],
    "subtheme":["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max", "subtheme_min", "subtheme_target", "subtheme_max_k"],
}


@dataclass(frozen=True)
class Config:
    seed: int
    cache_version: str
    data_url: str
    text_cols: tuple
    short_labels: tuple
    embed_backend: str
    embed_model: str
    embed_dim: int
    embed_batch: int
    umap_neighbors: int
    umap_min_dist: float
    cluster_dim: int
    k_mode: str
    k_fixed: int
    k_min: int
    k_max: int
    subtheme_min: int
    subtheme_target: int
    subtheme_max_k: int
    n_seeds: int
    n_bootstrap: int
    trust_k: int
    llm_enable: bool
    llm_model: str
    llm_n_quotes: int
    llm_force_regen: bool
    award_year: int
    checkpoint_dir: str

    def fingerprint(self, tag, n_rows, extra=None):
        """Short, stable cache key for `tag`. Built from the settings relevant to
        that stage, the row count, and an optional content signature `extra`
        (e.g. a hash of the actual input), so the cache is genuinely content
        addressed and never returns a stale object."""
        keys = _RELEVANT.get(tag, sorted(asdict(self).keys()))
        payload = {k: getattr(self, k) for k in keys}
        payload["n_rows"] = int(n_rows)
        payload["cache_version"] = self.cache_version
        if extra is not None:
            payload["extra"] = str(extra)
        blob = json.dumps(payload, sort_keys=True, default=str)
        return hashlib.md5(blob.encode()).hexdigest()[:10]


CFG = Config(
    seed=SEED, cache_version=CACHE_VERSION, data_url=DATA_URL, text_cols=TEXT_COLS, short_labels=SHORT_LABELS,
    embed_backend=EMBED_BACKEND, embed_model=_EMBED_MODEL, embed_dim=_EMBED_DIM,
    embed_batch=EMBED_BATCH, umap_neighbors=UMAP_NEIGHBORS, umap_min_dist=UMAP_MIN_DIST,
    cluster_dim=CLUSTER_DIM, k_mode=K_MODE, k_fixed=K_FIXED, k_min=K_MIN, k_max=K_MAX,
    subtheme_min=SUBTHEME_MIN, subtheme_target=SUBTHEME_TARGET, subtheme_max_k=SUBTHEME_MAX_K, n_seeds=N_SEEDS, n_bootstrap=N_BOOTSTRAP, trust_k=TRUST_K,
    llm_enable=LLM_ENABLE, llm_model=LLM_MODEL, llm_n_quotes=LLM_N_QUOTES,
    llm_force_regen=LLM_FORCE_REGEN, award_year=AWARD_YEAR,
    checkpoint_dir=CHECKPOINT_DIR,
)


def set_all_seeds(seed=None):
    """Fix every random source the pipeline uses. Call at the top of any stage
    that draws random numbers (UMAP, bootstrap, sampling)."""
    s = CFG.seed if seed is None else int(seed)
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


set_all_seeds()
print("=" * 64)
print("  STAGE 3  CONTROL PANEL")
print("=" * 64)
print(f"  seed            : {CFG.seed}")
print(f"  embedding       : {CFG.embed_model}  ({CFG.embed_dim}-d, backend={CFG.embed_backend})")
print(f"  UMAP            : n_neighbors={CFG.umap_neighbors}, min_dist={CFG.umap_min_dist}, "
      f"cluster_dim={CFG.cluster_dim}  (seeded, standard)")
print(f"  K (legacy knob) : mode={CFG.k_mode}, range=[{CFG.k_min}, {CFG.k_max}]  -- NOT the analytical K")
print(  "  analytical K    : 3-level hierarchy Main/Mid/Sub set in NB2 Stage 21-23c (bootstrap-Jaccard within interpretability bands); this legacy range is unused")
print(f"  LLM labelling   : {CFG.llm_model if CFG.llm_enable else 'disabled'}")
print(f"  cache fingerprint demo (embed, n=900) : {CFG.fingerprint('embed', 900)}")
print("=" * 64)


## Stage 4 — Library

Define-once utilities used everywhere: the content-hash cache `cached(tag, key, compute)` that replaces a manual fresh/load switch, the named result store `psave`/`pload`, the house style, the colour-blind-safe `palette`, and `save_fig`, which writes a 300 dpi PNG and a vector SVG for every figure.

In [ ]:
# @title
# ==============================================================================
# STAGE 4  -  LIBRARY  (define once, used everywhere)  [rewritten, storage-aware]
# Same house style + helpers as the monolith, with upgrades:
#   1. psave / pload / cached write LOCAL and MIRROR (<= size cap), read local
#      first then mirror, so artifacts survive a runtime reset and Notebook 2/3
#      find them automatically.
#   2. Every save is logged so write_manifest() can stamp a reproducible manifest.
#   3. artifact_diagnostic() reports which artifacts are present / missing.
#   4. validate_columns() is completed (monolith truncated it) and checks
#      Participant / Team.
#   5. invalidate() deletes an artifact so its stage recomputes next run.
# ==============================================================================
import os, pickle, hashlib, json, time, shutil
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys

POOLED_DIR = os.path.join(CFG.checkpoint_dir, "pooled_combined")
os.makedirs(POOLED_DIR, exist_ok=True)

MIRROR_POOLED_DIR = None
if "MIRROR_ROOT" in globals() and MIRROR_ROOT:
    MIRROR_POOLED_DIR = os.path.join(MIRROR_ROOT, "pooled_combined")
    try:
        os.makedirs(MIRROR_POOLED_DIR, exist_ok=True)
    except Exception:
        MIRROR_POOLED_DIR = None

_ARTIFACT_LOG = {}


def _cap_mb():
    return float(globals().get("ARTIFACT_SIZE_CAP_MB", 100))


def _mirror_file(path_local, mirror_dir):
    size_mb = os.path.getsize(path_local) / 1e6
    if not mirror_dir:
        return False, size_mb
    if size_mb > _cap_mb():
        print(f"    (LOCAL-ONLY: {os.path.basename(path_local)} = {size_mb:.0f} MB > {_cap_mb():.0f} MB cap)")
        return False, size_mb
    try:
        shutil.copy2(path_local, os.path.join(mirror_dir, os.path.basename(path_local)))
        return True, size_mb
    except Exception as e:
        print(f"    (mirror failed for {os.path.basename(path_local)}: {type(e).__name__})")
        return False, size_mb


def _log_artifact(name, path, mirrored):
    _ARTIFACT_LOG[name] = dict(path=path, bytes=int(os.path.getsize(path)),
                               size_mb=round(os.path.getsize(path) / 1e6, 3),
                               mirrored=bool(mirrored))


def cached(tag, key, compute):
    fname = f"{tag}_{key}.pkl"
    local = os.path.join(CFG.checkpoint_dir, fname)
    if os.path.exists(local):
        with open(local, "rb") as f:
            obj = pickle.load(f)
        print(f"  [{tag}] loaded from local cache  ({fname})")
        return obj
    if MIRROR_ROOT:
        mp = os.path.join(MIRROR_ROOT, fname)
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    obj = pickle.load(f)
                print(f"  [{tag}] restored from mirror  ({fname})")
                return obj
            except Exception:
                pass
    print(f"  [{tag}] computing (no cache for key {key}) ...")
    obj = compute()
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_ROOT)
    _log_artifact(tag, local, m)
    print(f"  [{tag}] saved  ({fname})")
    return obj


def psave(name, obj):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_POOLED_DIR)
    _log_artifact(name, local, m)


def pload(name):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(local):
        with open(local, "rb") as f:
            return pickle.load(f)
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    return pickle.load(f)
            except Exception:
                pass
    return None


save_ckpt = psave
def load_ckpt(name):
    return pload(name)


def invalidate(name):
    """Delete a named artifact (local + mirror) and any content-hash cache tagged
    with it, so the producing stage recomputes next run. The monolith referenced
    this helper but never defined it; provided here."""
    import glob as _glob
    removed = []
    for d in (POOLED_DIR, MIRROR_POOLED_DIR):
        if not d:
            continue
        p = os.path.join(d, f"{name}.pkl")
        if os.path.exists(p):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    for d in (CFG.checkpoint_dir, (MIRROR_ROOT if "MIRROR_ROOT" in globals() else None)):
        if not d:
            continue
        for p in _glob.glob(os.path.join(d, f"{name}_*.pkl")):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    _ARTIFACT_LOG.pop(name, None)
    print(f"  invalidate('{name}'): removed {len(removed)} file(s)")
    return removed


def data_sig(x):
    h = hashlib.md5()
    if isinstance(x, np.ndarray):
        h.update(np.ascontiguousarray(x).view(np.uint8)[::max(1, x.size // 100000)].tobytes())
        h.update(str(x.shape).encode())
    else:
        joined = "\x1f".join(map(str, x))
        h.update(joined.encode("utf-8", "ignore"))
    return h.hexdigest()[:10]


# ---------- house style ----------
# neutrals aligned with the NB3 design system (softer ink, cooler grid)
NAT_BG, NAT_TEXT, NAT_GRID = "#FFFFFF", "#14202B", "#E7EAEE"
mpl.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.edgecolor": "white",
    "axes.facecolor": "#FCFCFE", "axes.edgecolor": "#D9D9E0", "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.labelcolor": "#333333", "axes.labelsize": 9.5,
    "xtick.color": "#666666", "ytick.color": "#666666",
    "axes.axisbelow": True, "grid.color": "#EEEEF2", "grid.linewidth": 0.8,
    "legend.frameon": False, "font.size": 10, "figure.titleweight": "bold",
})
# ---------------------------------------------------------------------------
# UNIFIED SOFT-MODERN COLOUR SYSTEM (round 2). One system across NB1/NB2/NB3.
# Categorical set derived from Paul Tol's "muted" qualitative scheme
# (colourblind-validated; Tol 2021) - lower saturation than the old VIVID reds,
# no yellow on white, and the SAME hue keeps the same meaning in every figure.
# Semantic tokens (C_GOOD/C_WARN/C_BAD/C_NEUT) replace ad-hoc hexes in cells.
# ---------------------------------------------------------------------------
HIGHLIGHT = "#882255"          # soft wine accent (replaces the harsh maroon)
VIVID = ["#44AA99", "#CC6677", "#6699CC", "#AA4499", "#D9A44E", "#117733",
         "#882255", "#332288", "#5E81AC", "#B07AA1", "#996B4F", "#7A9E7E",
         "#C98D66", "#4C6B8A", "#8A7CA8", "#6FAF9F"]
C_GOOD, C_WARN, C_BAD, C_NEUT = "#44AA99", "#D9A44E", "#CC6677", "#9AA5B1"
try:
    import colorcet as _cc
    _GLASBEY = list(_cc.glasbey_category10)
except Exception:
    _GLASBEY = []


def _not_yellow(c):
    r, g, b = mcolors.to_rgb(c)
    h, sat, v = colorsys.rgb_to_hsv(r, g, b)
    return not (0.11 <= h <= 0.22 and sat > 0.30)


def apply_house_style():
    mpl.rcParams.update({
        "figure.facecolor": NAT_BG, "axes.facecolor": NAT_BG,
        "savefig.facecolor": NAT_BG, "axes.edgecolor": NAT_GRID,
        "axes.grid": True, "grid.color": NAT_GRID, "grid.linewidth": 0.6,
        "axes.axisbelow": True, "axes.titleweight": "bold",
        "text.color": NAT_TEXT, "axes.labelcolor": NAT_TEXT,
        "xtick.color": NAT_TEXT, "ytick.color": NAT_TEXT,
        "font.size": 11, "figure.dpi": 110, "savefig.dpi": 300,
    })


def palette(n):
    pool = list(VIVID)
    if n > len(pool):
        for c in _GLASBEY:
            hx = mcolors.to_hex(c)
            if _not_yellow(hx) and hx not in pool:
                pool.append(hx)
    if len(pool) < n:
        pool = pool * (n // len(pool) + 1)
    return [mcolors.to_hex(c) for c in pool[:n]]


def lighten(hex_color, amount):
    r, g, b = mcolors.to_rgb(hex_color)
    return mcolors.to_hex((r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount))


Q_COLORS = dict(zip(CFG.short_labels, palette(len(CFG.short_labels))))
# magnitude ramp: soft teal sequential (replaces the aggressive white->ruby ramp;
# same ramp NB3 uses, so heatmap intensity reads identically across the report)
GRAD = mcolors.LinearSegmentedColormap.from_list(
    "teal_seq", ["#F3FAF7", "#CDEDE1", "#8FD9BE", "#3FB98D", "#1D9E75", "#0F6E56"])
# publication-standard static export: SVG keeps TEXT AS TEXT (editable in
# Illustrator/Inkscape/Affinity), instead of outlining every glyph to paths.
mpl.rcParams["svg.fonttype"] = "none"


def save_fig(fig, name, interactive=False, static=True):
    base = os.path.join(POOLED_DIR, name)
    wrote = []
    if hasattr(fig, "write_html"):
        if interactive:
            fig.write_html(base + ".html", include_plotlyjs="cdn", full_html=True)
            wrote.append("HTML")
        if static:
            try:
                fig.write_image(base + ".png", scale=2)
                fig.write_image(base + ".svg")
                wrote += ["PNG", "SVG"]
            except Exception as e:
                print(f"  (Plotly static export needs kaleido; skipped: {type(e).__name__})")
    else:
        fig.savefig(base + ".png", dpi=300, bbox_inches="tight", facecolor=NAT_BG)
        fig.savefig(base + ".svg", bbox_inches="tight", facecolor=NAT_BG)
        wrote += ["PNG", "SVG"]
    for ext in (".png", ".svg", ".html"):
        p = base + ext
        if os.path.exists(p) and MIRROR_POOLED_DIR:
            _mirror_file(p, MIRROR_POOLED_DIR)
    try:
        from IPython.display import display as _display
        _display(fig)
    except Exception:
        pass
    if wrote:
        print(f"  saved figure: {os.path.basename(base)}  ({' + '.join(wrote)})")
    return base


def validate_columns(df):
    missing = [c for c in CFG.text_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Survey is missing expected text columns: {missing}. "
                       "Check DATA_URL / the export; the pipeline will not guess column names.")
    for c in ("Participant", "Team"):
        if c not in df.columns:
            raise KeyError(f"Survey is missing required column '{c}' "
                           "(needed for embedding metadata and the team-award join).")


def _now():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def write_manifest(filename, fingerprints, n_rows):
    man = dict(
        notebook=NOTEBOOK_NAME, notebook_version=NOTEBOOK_VERSION, timestamp=_now(),
        seed=CFG.seed, cache_version=CFG.cache_version, embed_model=CFG.embed_model,
        n_rows=int(n_rows), config_fingerprint=CFG.fingerprint("all", n_rows),
        fingerprints=dict(fingerprints),
        artifact_store=dict(local=POOLED_DIR, mirror=MIRROR_POOLED_DIR, size_cap_mb=_cap_mb()),
        artifacts=[dict(name=k, **v) for k, v in sorted(_ARTIFACT_LOG.items())],
    )
    p_local = os.path.join(CFG.checkpoint_dir, filename)
    with open(p_local, "w") as f:
        json.dump(man, f, indent=2, default=str)
    if MIRROR_ROOT:
        try:
            shutil.copy2(p_local, os.path.join(MIRROR_ROOT, filename))
        except Exception:
            pass
    print(f"  manifest written: {filename}  ({len(man['artifacts'])} artifacts logged, "
          f"config={man['config_fingerprint']})")
    return man


def artifact_status(name):
    lp = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(lp):
        return "local", os.path.getsize(lp) / 1e6
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            return "mirror", os.path.getsize(mp) / 1e6
    return "MISSING", 0.0


def artifact_diagnostic(required, optional=()):
    print("=" * 64)
    print("  ARTIFACT DIAGNOSTIC")
    print("=" * 64)
    print(f"  {'artifact':34s} {'where':8s} {'MB':>9}")
    print("  " + "-" * 54)
    missing = []
    for n in required:
        where, mb = artifact_status(n)
        if where == "MISSING":
            missing.append(n)
        flag = "" if where != "MISSING" else "   <-- MISSING"
        print(f"  {n:34s} {where:8s} {mb:9.2f}{flag}")
    for n in optional:
        where, mb = artifact_status(n)
        print(f"  {n:34s} {where:8s} {mb:9.2f}   (optional)")
    print("  " + "-" * 54)
    if missing:
        print(f"  {len(missing)} REQUIRED artifact(s) MISSING: {missing}")
    else:
        print("  all required artifacts present.")
    print("=" * 64)
    return missing


print("  Stage 4 library ready: storage-aware cache + psave/pload + manifest + diagnostics.")


## Stage 5 — Compatibility bridge

A thin bridge that exposes the names the analysis stages were written against, sourced from `CFG`, so the pipeline keeps one source of truth and needs no run-mode anywhere.

In [ ]:
# @title
# ==============================================================================
# STAGE 5  -  b  -  COMPATIBILITY BRIDGE
# The analysis stages were written against an earlier set of bare constant names.
# Rather than touch every one, we expose those names here, sourced from the one
# Control Panel (CFG) so there is still a single source of truth. Cleaning and
# outlier thresholds that were never clustering-critical live here as plain
# constants. There is no RUN_MODE: caching is automatic (Stage 4).
# ==============================================================================
# Core identifiers from CFG
SEED            = CFG.seed
DATA_URL        = CFG.data_url
TEXT_COLS       = list(CFG.text_cols)
SHORT_LABELS    = list(CFG.short_labels)
ONE_WORD_COL    = TEXT_COLS[4]
EMBED_MODEL     = CFG.embed_model
EMBED_DIM       = CFG.embed_dim
EMBED_BATCH     = CFG.embed_batch
TRUST_K         = CFG.trust_k
UMAP_QUALITY_METRIC = "euclidean"

# Clustering names used by the K-selection / Ward / validation cells
AGGLO_LINKAGE   = "ward"
AGGLO_METRIC    = "euclidean"
AGGLO_K_RANGE   = (CFG.k_min, CFG.k_max)
AGGLO_K_OVERRIDE = CFG.k_fixed if CFG.k_mode == "fixed" else None

# Language-model + awards
LLM_ENABLE      = CFG.llm_enable
LLM_MODEL_NAME  = CFG.llm_model
LLM_N_QUOTES    = CFG.llm_n_quotes
LLM_FORCE_REGEN = CFG.llm_force_regen
AWARD_TSV       = __import__("os").path.join(CFG.checkpoint_dir, "team_awards.tsv")
AWARD_YEAR      = CFG.award_year
AWARD_TEAMID_OVERRIDES = {"GreatBay_SCIE": "GreatBay-SCIE"}
MEDAL_RANK      = {"Gold": 3, "Silver": 2, "Bronze": 1, "None": 0}

# Colours: reuse the house palette so the old names resolve to the new style
# Capture the Stage 4 palette FUNCTION under a private name: several analysis
# cells reassign a local variable called `palette` to a list, so get_cluster_colors
# must not depend on the mutable global of that name.
_PALETTE_FN     = palette
NAT_COLORS      = _PALETTE_FN(len(SHORT_LABELS))
HIGHLIGHT_COLOR = HIGHLIGHT
def get_cluster_colors(n):
    return _PALETTE_FN(n)

# Text-cleaning / keyword sanitation constants (rarely tuned)
MIN_TOKEN_LENGTH = 2
HELSINKI_MODEL   = "Helsinki-NLP/opus-mt-mul-en"
KEYWORD_LATIN_ONLY = True
WC_DOMAIN_STOPS = {
    "igem", "team", "project", "experience", "nan", "thing", "lot", "way", "day",
    "time", "people", "part", "year", "place", "point", "fact", "kind", "group",
    "number", "case", "end", "question", "item", "work", "teammate", "new",
}
WC_SHORT_WHITELIST = {"lab", "phd", "dna", "rna", "fun", "pcr", "gel", "gmo",
                      "bio", "aid", "gap", "hub", "key", "aim"}
EXTRA_MULTILANG_STOPS = {
    "avant", "apres", "pendant", "environ", "depuis", "aprovado", "tudo", "tambm",
    "jamais", "aller", "venir", "passer", "rester", "devenir", "thing", "things",
    "something", "someone", "everyone", "everything", "really", "actually",
    "definitely", "probably", "maybe",
}

# Response-level outlier thresholds (Stage 10)
OUTLIER_MIN_ALPHA_RATIO  = 0.5
OUTLIER_MIN_ALPHA_TOKENS = 1
OUTLIER_MIN_VOWEL_RATIO  = 0.15
OUTLIER_MAX_INVALID_Q    = 3
SAME_WORD_THRESHOLD      = 3

print("  Stage 4b bridge ready: old config names resolve to CFG (no RUN_MODE).")


## Stage 6 — Imports

Shared third-party imports used across the analysis stages.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 6  -  C3_IMPORTS
# INPUT:  -
# OUTPUT: all libraries available
# ════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import umap as umap_lib
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from scipy.stats import spearmanr
from scipy.stats import spearmanr, ttest_ind, ttest_1samp
from scipy.spatial.distance import pdist
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from sklearn.manifold import trustworthiness
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
print('Imports complete.')

## Stage 7 — Data load

Downloads the survey once and caches it by data source; a re-run loads the same table. Column names are validated so a changed export fails loudly rather than mis-aligning.

In [ ]:
# @title
# ==============================================================================
# STAGE 7  -  DATA LOAD
# Downloads the survey once and caches it by the data source (Golden Rule 2): a
# re-run loads the same table instantly; pointing DATA_URL at a new file is the
# only thing that triggers a fresh download. Column names are then validated so a
# changed survey export fails loudly rather than silently mis-aligning (Rule 10).
# ==============================================================================
import gdown
import pandas as pd

def _download():
    gdown.download(CFG.data_url, "igem_experience_survey.csv", quiet=True)
    return pd.read_csv("igem_experience_survey.csv")

df = cached("data_raw", CFG.fingerprint("data", 0, extra=CFG.data_url), _download)
validate_columns(df)

print("=" * 64)
print("  STAGE 7  DATA LOAD")
print("=" * 64)
print(f"  responses : {len(df)}")
if "Team" in df.columns:
    print(f"  teams     : {df['Team'].nunique()}")
print(f"  columns   : {len(df.columns)}")
print("=" * 64)


## Stage 8 — Pre-clean: quality overview

A first look at the raw responses before cleaning: counts, completeness and obvious quality issues, so later changes can be judged against a known starting point.

In [ ]:
 # @title
# ════════════════════════════════════════════════════════
# STAGE 8  -  S1A_QUALITY_OVERVIEW
# INPUT:  df, TEXT_COLS, SHORT_LABELS, NAT_COLORS
# OUTPUT: chart_s1a_overview.png
# ════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Compute summary statistics ─────────────────────────────
summary_data = []

for col, short, color in zip(TEXT_COLS, SHORT_LABELS, NAT_COLORS):
    wc      = df[col].dropna().apply(lambda x: len(str(x).split()))
    missing = df[col].isna().sum()
    valid   = len(df) - missing
    n       = len(wc)
    summary_data.append({
        'short'      : short, 'color': color,
        'missing_pct': missing / len(df) * 100,
        'missing_n'  : missing,
        'valid_n'    : valid,
        'mean_wc'    : wc.mean(),
        'se_wc'      : wc.std() / np.sqrt(n),
        'sd_wc'      : wc.std(),
        'median_wc'  : wc.median(),
        'n'          : n,
    })
print('Summary statistics computed.')

# ── 2-panel chart ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 6.5),
                                gridspec_kw={'width_ratios': [1, 1.3]})
fig.patch.set_facecolor(NAT_BG)
plt.subplots_adjust(wspace=0.35)

y_pos = np.arange(len(SHORT_LABELS))[::-1]

# ── Panel A - Missing response rate ───────────────────────
ax1.set_facecolor(NAT_BG)
for d, y in zip(summary_data, y_pos):
    ax1.barh(y, d['missing_pct'], height=0.55, color=d['color'],
             alpha=0.80, edgecolor='white', zorder=3)
    ax1.text(d['missing_pct'] + 0.3, y,
             f"{d['missing_pct']:.1f}%  ({d['missing_n']})",
             va='center', fontsize=10, color=d['color'], fontweight='bold')

ax1.set_yticks(y_pos)
ax1.set_yticklabels([d['short'] for d in summary_data],
                    fontsize=11, fontweight='bold')
for tick, d in zip(ax1.get_yticklabels(), summary_data):
    tick.set_color(d['color'])
ax1.set_xlabel('Missing response rate (%)', fontsize=11, color=NAT_TEXT, labelpad=8)
ax1.set_title('A   Missing Response Rate',
              fontsize=12, fontweight='bold', color=NAT_TEXT, pad=12, loc='left')
ax1.set_xlim(0, max(d['missing_pct'] for d in summary_data) + 8)
ax1.xaxis.grid(True, color=NAT_GRID, linewidth=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines[['top', 'right']].set_visible(False)
ax1.spines[['left', 'bottom']].set_color(NAT_GRID)
ax1.tick_params(axis='y', length=0)
ax1.tick_params(axis='x', colors='#888888', labelsize=9)

# ── Panel B - Mean word count (lollipop + SE + SD) ────────
ax2.set_facecolor(NAT_BG)
for d, y in zip(summary_data, y_pos):
    # SD range - light transparent band
    ax2.barh(y, d['sd_wc'] * 2, left=d['mean_wc'] - d['sd_wc'],
             height=0.52, color=d['color'], alpha=0.12,
             edgecolor='none', zorder=2)
    # SE range - darker band
    ax2.barh(y, d['se_wc'] * 2, left=d['mean_wc'] - d['se_wc'],
             height=0.52, color=d['color'], alpha=0.45,
             edgecolor='none', zorder=3)
    # Stem line
    ax2.plot([0, d['mean_wc']], [y, y],
             color=d['color'], linewidth=1.8, alpha=0.35, zorder=2)
    # Mean dot
    ax2.scatter([d['mean_wc']], [y], color=d['color'],
                s=180, zorder=5, edgecolors='white', linewidths=2.5)
    # Mean label
    label_x = d['mean_wc'] + d['se_wc'] + 1.2
    ax2.text(label_x, y, f"{d['mean_wc']:.1f}",
             va='center', ha='left', fontsize=13,
             fontweight='bold', color=d['color'])
    ax2.text(label_x, y - 0.28, f"\u00b1{d['se_wc']:.2f} SE",
             va='top', ha='left', fontsize=8.5,
             color='#999999', style='italic')
    ax2.text(label_x + 6.5, y, f"n={d['n']}",
             va='center', ha='left', fontsize=9.5, color='#AAAAAA')

ax2.set_yticks(y_pos)
ax2.set_yticklabels([d['short'] for d in summary_data],
                    fontsize=11, fontweight='bold')
for tick, d in zip(ax2.get_yticklabels(), summary_data):
    tick.set_color(d['color'])
ax2.set_xlabel('Word count', fontsize=11, color=NAT_TEXT, labelpad=8)
ax2.set_title('B   Mean Response Length (\u00b1 SE / SD)',
              fontsize=12, fontweight='bold', color=NAT_TEXT, pad=12, loc='left')
ax2.set_xlim(-0.5, max(d['mean_wc'] + d['sd_wc'] + 12 for d in summary_data))
ax2.set_ylim(-0.6, len(SHORT_LABELS) - 0.4)
ax2.xaxis.grid(True, color=NAT_GRID, linewidth=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.spines['bottom'].set_color(NAT_GRID)
ax2.tick_params(axis='y', length=0)
ax2.tick_params(axis='x', colors='#888888', labelsize=9)
ax2.legend(handles=[
    mpatches.Patch(facecolor='#888', alpha=0.45, label='\u00b11 SE'),
    mpatches.Patch(facecolor='#888', alpha=0.12, label='\u00b11 SD'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#888',
               markersize=10, markeredgecolor='white', markeredgewidth=2,
               label='Mean'),
], loc='lower right', fontsize=9, framealpha=0.95,
   facecolor=NAT_BG, edgecolor=NAT_GRID)

fig.suptitle('S1 \u2014 Pre-Cleaning Data Quality Overview',
             fontsize=14, fontweight='bold', color=NAT_TEXT, y=1.01)
fig.text(0.5, -0.02,
         'Panel A: percentage of missing responses per question.  '
         'Panel B: mean word count with \u00b11 SE (dark) and \u00b11 SD (light).  '
         'Q5 (One Word) expected to be short by design.',
         ha='center', fontsize=8.5, color='#AAAAAA', style='italic')

plt.tight_layout()
# FIX: was a bare 180-dpi plt.savefig to CHECKPOINT_DIR - no SVG, no mirror,
# absent from the manifest. Routed through save_fig like every other figure.
save_fig(fig, 'chart_s1a_overview')
plt.close(fig)

## Stage 9 — Pre-clean: distributions

Distribution of response length and other descriptive views of the raw corpus.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 9  -  S1B_DISTRIBUTION
# INPUT:  df, TEXT_COLS, SHORT_LABELS
# OUTPUT: chart_s1b_distribution.png
# ════════════════════════════════════════════════════════
# Descriptive word count distributions per question.
# filters (alpha ratio, vowel ratio, copy-paste), not word counts.
# This chart is purely descriptive: shows response length patterns.
from matplotlib.lines import Line2D


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(NAT_BG)
plt.subplots_adjust(hspace=0.45, wspace=0.35)
ax_list = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1]]
axes[1,2].set_visible(False)

for ax, col, short, color in zip(ax_list, TEXT_COLS, SHORT_LABELS, NAT_COLORS):
    wc   = df[col].dropna().apply(lambda x: len(str(x).split()))
    vals = wc.values
    n    = len(vals)

    # Adaptive binning based on question type
    is_ow   = (col == ONE_WORD_COL)
    max_bin = 15 if is_ow else min(vals.max() + 5, 150)
    n_bins  = 15 if is_ow else 35
    bins    = np.linspace(0, max_bin, n_bins)

    # Single-color histogram - no threshold coloring
    ax.hist(vals, bins=bins, color=color, alpha=0.75,
            edgecolor='white', linewidth=0.5, zorder=3)

    # Descriptive statistics: mean + median
    mu  = vals.mean()
    med = np.median(vals)
    ax.axvline(mu, color=color, linewidth=2.0, linestyle='-',
               zorder=5, alpha=0.9)
    ax.axvline(med, color=color, linewidth=1.5, linestyle='--',
               zorder=5, alpha=0.7)

    # Stats annotation box
    q1, q3 = np.percentile(vals, [25, 75])
    ax.text(0.97, 0.95,
            f'n={n}\n'
            f'\u03bc={mu:.1f}\n'
            f'med={med:.0f}\n'
            f'IQR={q1:.0f}\u2013{q3:.0f}',
            transform=ax.transAxes, fontsize=9,
            va='top', ha='right', color='#555555',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#DDDDDD', linewidth=0.8, alpha=0.92))

    ax.set_title(short, fontsize=12, fontweight='bold', color=color, pad=8)
    ax.set_xlabel('Word count', fontsize=10, color=NAT_TEXT)
    ax.set_ylabel('Frequency', fontsize=10, color=NAT_TEXT)
    ax.yaxis.grid(True, color=NAT_GRID, linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(NAT_GRID)
    ax.tick_params(colors='#888888', labelsize=8.5)

    # Minimal legend
    ax.legend(handles=[
        Line2D([0], [0], color=color, linewidth=2.0, linestyle='-', label='Mean'),
        Line2D([0], [0], color=color, linewidth=1.5, linestyle='--', label='Median'),
    ], fontsize=8, framealpha=0.9, facecolor=NAT_BG, edgecolor=NAT_GRID,
       loc='upper right', bbox_to_anchor=(0.75, 1.0))

fig.suptitle('S1 \u2014 Pre-Cleaning: Word Count Distribution',
             fontsize=14, fontweight='bold', color=NAT_TEXT, y=1.01)
fig.text(0.5, -0.01,
         'Descriptive overview of response lengths.  '
         'Solid = mean  \u00b7  Dashed = median  \u00b7  '
         'Outlier detection uses content quality filters (S4A), not word count thresholds.',
         ha='center', fontsize=8.5, color='#AAAAAA', style='italic')
# FIX: was a bare 180-dpi plt.savefig to CHECKPOINT_DIR - no SVG, no mirror,
# absent from the manifest. Routed through save_fig like every other figure.
save_fig(fig, 'chart_s1b_distribution')
plt.close(fig)

## Stage 10 — Language detection

Language identification with Lingua, which is needed because the survey is multilingual; this informs the cleaning and the choice of a multilingual embedding model.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 10  -  S2A_LINGUA
# INPUT:  -
# OUTPUT: detect_language function
# ════════════════════════════════════════════════════════
# Detects ANY spoken language (74 supported by Lingua).
# Returns ISO 639-1 two-letter code (e.g., 'en', 'fr', 'ru', 'hi', 'ko').
# Self-contained: auto-installs lingua if missing (runtime-restart-safe).

import subprocess
import sys
import importlib

# ── Self-healing install check ────────────────────────────
def _ensure_package(pkg_import_name, pip_name=None):
    """Ensures a package is importable; installs via pip if missing."""
    try:
        importlib.import_module(pkg_import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            pip_name or pkg_import_name
        ])

_ensure_package('lingua', 'lingua-language-detector')
_ensure_package('pandas')

import pandas as pd
from lingua import LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_spoken_languages(
    ).with_preloaded_language_models().build()

def detect_language(text):
    """Return ISO 639-1 language code for any input text."""
    if pd.isna(text) or not str(text).strip():
        return 'en'
    try:
        result = detector.detect_language_of(str(text))
        if result is None:
            return 'en'
        return result.iso_code_639_1.name.lower()
    except Exception:
        return 'en'

# ── Sanity check ──────────────────────────────────────────
_test_cases = {
    'Hello world': 'en',
    'Bonjour tout le monde': 'fr',
    'Hola a todos': 'es',
    '你好世界': 'zh',
    'Привет мир': 'ru',
    'नमस्ते दुनिया': 'hi',
}
print('Lingua detector loaded (all spoken languages).')
for txt, expected in _test_cases.items():
    got = detect_language(txt)
    mark = 'OK' if got == expected else 'WARNING'
    print(f'  {mark} "{txt[:30]}" -> {got} (expected {expected})')

## Stage 11 — Outlier screening (before any cleaning)

Flags low-information responses with four transparent rules (copy-paste runs, too few alphabetic tokens, very low vowel ratio, repeated single words), CJK-aware so non-Latin answers are not wrongly removed.

In [ ]:
# @title
# ════════════════════════════════════════════════════════════════
# STAGE 15  -  S4A_OUTLIER_DETECT   (now runs BEFORE cleaning + embedding)
# INPUT:  df (RAW text columns), TEXT_COLS, SHORT_LABELS, OUTLIER_* thresholds
# OUTPUT: to_remove, valid_<question> flags, chart_s4a_outlier
# ════════════════════════════════════════════════════════════════
# ---------------------------------------------------------------------------
# ORDERING FIX (critical). This stage used to run AFTER the embedding stage, and
# the pooling stage consumed `embeddings_dict`, which had already been computed
# from the UNFILTERED frame. Net effect: outlier detection and removal ran, printed
# a confident diagnostic, wrote df_final... and then changed NOTHING downstream.
# Every flagged copy-paste / junk response was still embedded, still clustered,
# still counted in the themes. (The old Stage 17 then reassigned
# `df = load_ckpt('df_cleaned')`, quietly restoring the removed participants to
# the working frame as well.)
# This cell now runs on the RAW text BEFORE any cleaning or embedding, so the
# `valid_*` flags and `to_remove` list actually govern what gets embedded.
# It reads only raw survey columns, so it has no dependency on the cleaning stage.
# ---------------------------------------------------------------------------

import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter
import matplotlib

# ── Rule helpers ───────────────────────────────────────────────
def alpha_ratio(s):
    """Fraction of characters that are letters, in ANY script."""
    s = str(s)
    alpha = sum(c.isalpha() for c in s)
    return alpha / max(len(s), 1)

def vowel_ratio(s):
    """Latin-script vowel density. Only meaningful for Latin text: used as a
    keyboard-mash detector. Non-Latin scripts are exempted by the caller."""
    s = str(s).lower()
    alpha = [c for c in s if c.isalpha()]
    if not alpha:
        return 0.0
    return sum(c in 'aeiouàâäéèêëîïôùûü' for c in alpha) / len(alpha)

def is_cjk(s):
    return any('一' <= c <= '鿿' or
               '぀' <= c <= 'ヿ' or
               '가' <= c <= '힣' for c in str(s))

def is_latin_script(s):
    """True if the letters present are predominantly Latin. The vowel-ratio rule
    below is Latin-specific; applying it to Cyrillic/Greek/Arabic/Devanagari
    would flag perfectly good responses as keyboard mash."""
    letters = [c for c in str(s) if c.isalpha()]
    if not letters:
        return False
    latin = sum(1 for c in letters if ('a' <= c.lower() <= 'z') or 'À' <= c <= 'ɏ')
    return latin / len(letters) > 0.5

def alpha_token_count(s):
    if is_cjk(str(s)):
        return OUTLIER_MIN_ALPHA_TOKENS + 1  # CJK is not whitespace-tokenised; exempt
    return len([t for t in str(s).split() if any(ch.isalpha() for ch in t)])

# ── Load df if needed ─────────────────────────────────────────
# FIX: the old line was  df = load_ckpt('df_final') or load_ckpt('df_raw')
#   * `DataFrame or ...` raises "truth value of a DataFrame is ambiguous"
#     (NB2 hit this and added get_df(); NB1 never got the fix), and
#   * 'df_raw' is never saved under that name (the raw table lives in the
#     content-hash cache as 'data_raw'), so the fallback could not work anyway.
try:
    _ = df
except NameError:
    df = load_ckpt('df_cleaned')
    if df is None:
        raise RuntimeError('Run the data-load stage first (need the survey frame).')

# ── Track originally blank per question ──────────────────────
originally_blank = {}
for short in SHORT_LABELS:
    col = TEXT_COLS[SHORT_LABELS.index(short)]
    originally_blank[short] = df[col].isna() | (df[col].str.strip() == '')

# ── Initialize valid_ flags ───────────────────────────────────
for short in SHORT_LABELS:
    df[f'valid_{short}'] = True

# ── Rule 1: Copy-paste (same text in ≥4/5, ≤3 words) ─────────
copy_paste_participants = {}
for _, row in df.iterrows():
    texts = [str(row[c]).strip().lower() for c in TEXT_COLS
             if pd.notna(row[c]) and str(row[c]).strip()]
    if len(texts) < 4:
        continue
    most_common_text, freq = Counter(texts).most_common(1)[0]
    if freq >= 4 and len(most_common_text.split()) <= 3:
        copy_paste_participants[row['Participant']] = most_common_text

for short in SHORT_LABELS:
    col = TEXT_COLS[SHORT_LABELS.index(short)]
    for idx, row in df.iterrows():
        if row['Participant'] not in copy_paste_participants:
            continue
        repeated = copy_paste_participants[row['Participant']]
        val = str(row[col]).strip().lower() if pd.notna(row[col]) else ''
        if val == repeated:
            df.at[idx, f'valid_{short}'] = False

# ── Rule 2: Content quality ───────────────────────────────────
rule2_log = []
for short in SHORT_LABELS:
    col = TEXT_COLS[SHORT_LABELS.index(short)]
    for idx, row in df.iterrows():
        val = row[col]
        if pd.isna(val) or str(val).strip() == '':
            continue
        s = str(val).strip()
        ar = alpha_ratio(s)
        tc = alpha_token_count(s)
        # The vowel rule is a LATIN keyboard-mash heuristic. Guarding it by script
        # stops it from flagging valid Cyrillic / Greek / Arabic / Devanagari
        # answers (which have no a-e-i-o-u) as junk.
        vowel_fail = (is_latin_script(s) and vowel_ratio(s) < OUTLIER_MIN_VOWEL_RATIO
                      and len(s) > 8)
        fail = (ar < OUTLIER_MIN_ALPHA_RATIO or vowel_fail or
                tc < OUTLIER_MIN_ALPHA_TOKENS)
        if fail:
            df.at[idx, f'valid_{short}'] = False
            rule2_log.append((row['Participant'], short, val))

# ── Rule 4: Full removal - ALL 5 responses invalid/blank ─────
to_remove = []
for idx, row in df.iterrows():
    all_bad = all(
        (not row[f'valid_{short}'] or originally_blank[short].loc[idx])
        for short in SHORT_LABELS
    )
    if all_bad:
        to_remove.append(row['Participant'])

# ── Collect NaN'd responses (excl. fully-removed) ────────────
nannd_responses = []
for short in SHORT_LABELS:
    col = TEXT_COLS[SHORT_LABELS.index(short)]
    for idx, row in df.iterrows():
        if row['Participant'] in to_remove:
            continue
        if originally_blank[short].loc[idx]:
            continue
        if not row[f'valid_{short}']:
            nannd_responses.append((row['Participant'], short, row[col]))

# ═══════════════════════════════════════════════════════════════
# DIAGNOSTIC PRINT
# ═══════════════════════════════════════════════════════════════
print('=' * 70)
print('  S4A FULL DIAGNOSTIC - All Outlier Removal Actions')
print('=' * 70)
n_raw   = len(df)
n_fully = len(to_remove)
n_nannd = len(nannd_responses)
n_kept  = n_raw - n_fully
print(f'  Total participants in raw data    : {n_raw}')
print(f'  Participants retained             : {n_kept}')
print(f'  Participants fully removed        : {n_fully}')
print(f'  Individual responses NaN\'d        : {n_nannd}')
print('=' * 70)

print()
print('─' * 70)
print('  PART 1 - Individual Responses NaN\'d (Rules 1 & 2)')
n_pids = len(set(r[0] for r in nannd_responses))
print(f'  Count: {n_nannd} responses across {n_pids} participants')
print('─' * 70)

by_pid = defaultdict(list)
for pid, short, text in nannd_responses:
    by_pid[pid].append((short, text))

for pid, entries in by_pid.items():
    team = df.loc[df['Participant'] == pid, 'Team'].values[0]
    print(f'\n  Participant : {pid[:27]}...')
    print(f'  Team        : {team}')
    print(f'  Affected    : {len(entries)} of 5 responses NaN\'d')
    for short, text in entries:
        print(f'    [{short:<12}] "{str(text)[:80]}"')

print()
print('─' * 70)
print('  PART 2 - Fully Removed Participants (Rule 4)')
print(f'  Count: {len(to_remove)} participants')
print('─' * 70)
for pid in to_remove:
    team = df.loc[df['Participant'] == pid, 'Team'].values[0]
    print(f'\n  Participant : {pid[:27]}...')
    print(f'  Team        : {team}')
    for short in SHORT_LABELS:
        col = TEXT_COLS[SHORT_LABELS.index(short)]
        val = df.loc[df['Participant'] == pid, col].values[0]
        print(f'    [{short:<12}] {val}')

print()
print('─' * 70)
print('  Per-Question Impact Summary')
print('─' * 70)
print(f'  {"Question":<16} {"NaN":>8} {"Retained":>10} {"% retained":>11}')
print(f'  {"──────────────":<16} {"────────":>8} {"──────────":>10} {"──────────":>11}')
for short in SHORT_LABELS:
    n_nannd_q = sum(1 for pid, s, _ in nannd_responses if s == short)
    non_removed = df[~df['Participant'].isin(to_remove)]
    n_retained = int(
        non_removed[f'valid_{short}'].sum()
        - originally_blank[short].loc[non_removed.index].sum()
    )
    pct = 100.0 * n_retained / n_kept if n_kept else 0
    print(f'  {short:<16} {n_nannd_q:>8} {n_retained:>10} {pct:>10.1f}%')

print()
print('=' * 70)
print('  End of full diagnostic review.')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════
# CHART: chart_s4a_outlier.png  (redesigned - no overlap, authentic values)
# ═══════════════════════════════════════════════════════════════
from matplotlib.patches import Patch

# ── Per-question composition (retained / NaN'd / blank), authentic values ──
non_removed = df[~df['Participant'].isin(to_remove)]
nidx = non_removed.index
per_q_comp = {}
for short in SHORT_LABELS:
    n_blank   = int(originally_blank[short].loc[nidx].sum())
    n_nan_q   = sum(1 for pid, s, _ in nannd_responses if s == short)
    n_present = len(non_removed) - n_blank
    n_ret     = n_present - n_nan_q
    per_q_comp[short] = (max(n_ret, 0), n_nan_q, n_blank)

maxtot = max(sum(v) for v in per_q_comp.values()) if per_q_comp else 1
plt.rcParams['font.family'] = 'sans-serif'
fig = plt.figure(figsize=(15.5, 6.6)); fig.patch.set_facecolor(NAT_BG)
grid = fig.add_gridspec(1, 2, width_ratios=[1.6, 1.0], wspace=0.30)
C_RET, C_NAN, C_BLK = '#44AA99', '#D9A44E', '#D8DCE0'

axA = fig.add_subplot(grid[0, 0]); axA.set_facecolor(NAT_BG)
ys = np.arange(len(SHORT_LABELS))[::-1]
for i, short in enumerate(SHORT_LABELS):
    ret, nan, blk = per_q_comp[short]
    total = ret + nan + blk if (ret + nan + blk) else 1
    y = ys[i]
    axA.barh(y, ret, color=C_RET, height=0.62, zorder=3, edgecolor='white', lw=0.7)
    axA.barh(y, nan, left=ret, color=C_NAN, height=0.62, zorder=3, edgecolor='white', lw=0.7)
    axA.barh(y, blk, left=ret + nan, color=C_BLK, height=0.62, zorder=3, edgecolor='white', lw=0.7)
    axA.text(ret / 2, y, f'{ret}', ha='center', va='center',
             fontsize=10, fontweight='bold', color='white', zorder=4)
    if nan > 0:
        axA.text(ret + nan / 2, y + 0.40, f'{nan}', ha='center', va='bottom',
                 fontsize=7.5, color='#B07A00', zorder=4)
    axA.text(maxtot * 1.07, y, f'{100 * ret / total:.0f}% kept', va='center',
             fontsize=9.5, color=NAT_TEXT, fontweight='bold')
axA.set_yticks(ys); axA.set_yticklabels(SHORT_LABELS, fontsize=10.5, color=NAT_TEXT)
axA.set_xlabel('Responses', fontsize=10, color=NAT_TEXT)
axA.set_title('A · Per-question response composition after cleaning',
              fontsize=11.5, fontweight='bold', color=NAT_TEXT, pad=26, loc='left')
axA.set_xlim(0, maxtot * 1.20)
axA.spines[['top', 'right']].set_visible(False)
axA.spines[['left', 'bottom']].set_color(NAT_GRID)
axA.tick_params(length=0)
axA.xaxis.grid(True, color='#F2F2F2', lw=0.8); axA.set_axisbelow(True)
axA.legend(handles=[Patch(color=C_RET, label='Retained (valid)'),
                    Patch(color=C_NAN, label="NaN'd (failed quality rules)"),
                    Patch(color=C_BLK, label='Originally blank')],
           fontsize=8.8, loc='lower left', bbox_to_anchor=(0, 1.005),
           framealpha=0, edgecolor='none', ncol=3, handlelength=1.3, columnspacing=1.4)

axB = fig.add_subplot(grid[0, 1]); axB.set_facecolor(NAT_BG)
cats = ['Raw\nparticipants', 'Retained', 'Fully\nremoved']
vals = [n_raw, n_kept, n_fully]
cols = ['#9AA5B1', C_RET, '#CC6677']
bars = axB.bar(range(3), vals, color=cols, width=0.6, zorder=3, edgecolor='white', lw=1)
for b, v in zip(bars, vals):
    axB.text(b.get_x() + b.get_width() / 2, v + n_raw * 0.012, f'{v}',
             ha='center', va='bottom', fontsize=11, fontweight='bold', color=NAT_TEXT)
axB.set_xticks(range(3)); axB.set_xticklabels(cats, fontsize=9.5, color=NAT_TEXT)
axB.set_ylabel('Participants', fontsize=10, color=NAT_TEXT)
axB.set_title(f'B · Participant retention (N={n_raw})',
              fontsize=11.5, fontweight='bold', color=NAT_TEXT, pad=26, loc='left')
axB.set_ylim(0, n_raw * 1.18)
axB.spines[['top', 'right']].set_visible(False)
axB.spines[['left', 'bottom']].set_color(NAT_GRID)
axB.tick_params(length=0)
axB.yaxis.grid(True, color='#F2F2F2', lw=0.8); axB.set_axisbelow(True)
axB.text(0.62, 0.55, f"+{n_nannd} individual\nresponses NaN'd\n(retained participants)",
         transform=axB.transAxes, ha='left', va='center', fontsize=8.6,
         color='#777', style='italic',
         bbox=dict(boxstyle='round,pad=0.45', facecolor='#F7F8F9', edgecolor=NAT_GRID, lw=0.6))

fig.suptitle('S4A · Content-Quality Outlier Screening',
             fontsize=14, fontweight='bold', color=NAT_TEXT, y=1.04)
fig.text(0.5, -0.02,
         'Rules: copy-paste detection (same short text in \u22654/5 answers) · '
         'alpha-ratio \u22650.5 · vowel-ratio \u22650.15 (len>8) · \u22651 alpha token (CJK exempt).\n'
         'A participant is fully removed only if all 5 answers are blank or fail. No word-count threshold.',
         ha='center', fontsize=8.3, color='#999', style='italic')

# round 2: routed through save_fig (300 dpi PNG + editable-text SVG + mirror +
# manifest) like every other figure; was a bare 180-dpi PNG to CHECKPOINT_DIR.
save_fig(fig, 'chart_s4a_outlier')
plt.close(fig)

## Stage 12 — Outlier removal

Removes the flagged participants and saves the final table. Deterministic and recomputed each run.

In [ ]:
# @title
# ==============================================================================
# STAGE 16  -  b  -  OUTLIER REMOVAL
# Drops the participants flagged by the 4-rule detector (Stage 10a) and saves
# df_final. Deterministic; recomputed each run.
# ==============================================================================
n_before = len(df)
if to_remove:
    df = df[~df['Participant'].isin(to_remove)].copy().reset_index(drop=True)
n_after   = len(df)
n_removed = n_before - n_after
# interim save: at this point df is screened but NOT yet cleaned (no bert_*).
# The cleaning stage re-saves df_final with the text columns NB2 needs.
save_ckpt('df_final', df)

print('=' * 55)
print('  STAGE 16B - OUTLIER REMOVAL SUMMARY')
print('=' * 55)
print(f'  Filter         : 4-rule response-level detection (copy-paste + quality + CJK-aware)')
print(f'  Original       : {n_before} participants')
print(f'  Removed        : {n_removed} participants')
print(f'  Final dataset  : {n_after} participants')
print(f'  Retention rate : {n_after/n_before*100:.1f}%')
if n_removed == 0:
    print('  No participants removed - dataset quality is high.')


## Stage 13 — Cleaning functions

The text-normalisation functions. `minimal_clean` keeps text close to the original for embedding (a multilingual model handles casing and punctuation); `heavy_clean` produces a sanitised form for keyword counting.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 11  -  S2B_CLEANING_FUNCS
# INPUT:  HELSINKI_MODEL
# OUTPUT: minimal_clean / heavy_clean functions
# ════════════════════════════════════════════════════════
# Two cleaning pipelines:
#   minimal_clean() -> bert_* columns -> EMBEDDING input
#     No lemmatization, preserves natural language structure.
#   heavy_clean() -> clean_* columns -> word cloud / TF-IDF input
#     Q1-Q4: POS-filtered (NOUN/PROPN/ADJ only), lemmatized, non-EN excluded.
#     Q5 (One Word): minimal processing only - no POS, no lemmatization.
# Reference: Santos et al. (2025); Kilgarriff (2001).
#
# ---------------------------------------------------------------------------
# BUG FIX (critical, silent): the previous minimal_clean() used the character
# class  [^a-zA-Z一-鿿À-ɏ\s]  which keeps ONLY Latin, Latin-1
# supplement/extended, and CJK *ideographs*. Every other script was deleted
# outright. Verified effect on real inputs:
#     Russian  "Привет мир команда"      -> ''        (row becomes NaN, DROPPED)
#     Hindi    "नमस्ते दुनिया"                -> ''        (DROPPED)
#     Korean   "팀워크가 중요합니다"          -> ''        (DROPPED)
#     Greek    "Η ομάδα ήταν καλή"       -> ''        (DROPPED)
#     Arabic   "العمل الجماعي مهم"         -> ''        (DROPPED)
#     Japanese "チームワークは重要です"       -> '重要'    (MANGLED: kana stripped,
#                                                    only the kanji survive, so a
#                                                    2-char fragment gets embedded
#                                                    AS IF it were the response)
# This silently contradicted the rest of the pipeline, which advertises
# multilingual support (Lingua detects 74 languages; Qwen3-Embedding and BGE-M3
# are multilingual and handle these scripts natively). Non-Latin respondents were
# being deleted from the study, or worse, misrepresented by a fragment.
#
# The fix: strip by UNICODE CATEGORY, not by a Latin/CJK allow-list. We remove
# punctuation, symbols and digits, and keep any letter or mark in ANY script.
# This is what the multilingual embedder actually wants.
# ---------------------------------------------------------------------------

import unicodedata
import spacy
import contractions as contractions_lib
from transformers import MarianMTModel, MarianTokenizer

nlp = spacy.load('en_core_web_sm')
print('spaCy loaded.')

print(f'Loading {HELSINKI_MODEL}...')
try:
    trans_tokenizer = MarianTokenizer.from_pretrained(HELSINKI_MODEL)
    trans_model     = MarianMTModel.from_pretrained(HELSINKI_MODEL)
    USE_TRANSLATOR  = True
    print('Helsinki-NLP loaded.')
except Exception as e:
    print(f'Helsinki-NLP unavailable: {e}')
    USE_TRANSLATOR = False

def translate_to_english(text):
    """Translate non-English text to English via Helsinki-NLP.
    Returns original text if translation fails."""
    if not USE_TRANSLATOR:
        return text
    try:
        inputs     = trans_tokenizer(str(text), return_tensors='pt',
                                     padding=True, truncation=True, max_length=512)
        translated = trans_model.generate(**inputs)
        result     = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
        if not result.strip() or result.strip().lower() == str(text).strip().lower():
            return text
        return result
    except Exception:
        return str(text)


def _strip_non_letters(text):
    """Keep letters/marks of ANY script + whitespace; drop punctuation, symbols,
    digits. Unicode-category based, so no script is privileged (fixes the
    Latin/CJK-only allow-list that deleted Cyrillic/Devanagari/Hangul/kana/
    Greek/Arabic and mangled Japanese)."""
    out = []
    for ch in text:
        if ch.isspace():
            out.append(' ')
            continue
        cat = unicodedata.category(ch)      # L* = letter, M* = combining mark
        out.append(ch if cat[0] in ('L', 'M') else ' ')
    return ''.join(out)


def minimal_clean(text):
    """Lightweight cleaning for the EMBEDDING input.
    Preserves natural language structure - no lemmatization, script-agnostic.
    Reference: Santos et al. (2025); Devlin et al. (2019)."""
    if pd.isna(text) or not str(text).strip():
        return ''
    text = str(text).strip()
    lang = detect_language(text)
    if lang == 'en':
        text = contractions_lib.fix(text)
    text = text.lower()                     # no-op for uncased scripts; harmless
    text = _strip_non_letters(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# POS tags to keep in word clouds: content words only
KEEP_POS = {'NOUN', 'PROPN', 'ADJ'}

def heavy_clean(text, is_one_word_col=False):
    """Deep cleaning for word clouds and TF-IDF.
    Q1-Q4: POS filter (NOUN/PROPN/ADJ), lemmatized. Non-EN excluded.
    Q5 (One Word): translate -> lemmatize, no POS filter.
      spaCy cannot reliably POS-tag isolated words without sentence context.
    NOTE: this column feeds keyword counting ONLY. Dropping non-English here is a
    deliberate, documented choice (the English POS model cannot tag them); it is
    NOT a reason to drop them from the embedding, which is why minimal_clean is
    now script-agnostic.
    Reference: Kilgarriff (2001); Tiedemann & Thottingal (2020)."""
    if pd.isna(text) or not str(text).strip():
        return ''
    text = str(text).strip()
    lang = detect_language(text)

    if lang != 'en':
        if not is_one_word_col:
            return ''
        translated = translate_to_english(text)
        if detect_language(translated) != 'en':
            return ''
        text = translated

    text = minimal_clean(text)
    if not text:
        return ''

    if is_one_word_col:
        doc = nlp(text)
        tokens = [t.lemma_ for t in doc
                  if len(t.lemma_) >= MIN_TOKEN_LENGTH
                  and t.lemma_.lower() not in WC_DOMAIN_STOPS]
        return ' '.join(tokens)

    doc = nlp(text)
    tokens = [t.lemma_ for t in doc
              if len(t.lemma_) >= MIN_TOKEN_LENGTH
              and t.pos_ in KEEP_POS]
    return ' '.join(tokens)


# ---- self-check: the scripts that the old regex destroyed must now survive ----
_probe = {'Russian': 'Привет мир команда', 'Hindi': 'नमस्ते दुनिया',
          'Korean': '팀워크가 중요합니다', 'Japanese': 'チームワークは重要です',
          'Greek': 'Η ομάδα ήταν καλή', 'Arabic': 'العمل الجماعي مهم',
          'Chinese': '团队合作非常重要', 'English': 'The mentors were supportive!'}
print('Cleaning functions defined.')
print(f'  Q1-Q4 POS filter: keeping {KEEP_POS} only')
print(f'  Q5 (One Word): lemmatized, no POS filter; non-EN translated first')
print('  multilingual self-check (minimal_clean must NOT empty these):')
_bad = []
for _lang, _t in _probe.items():
    _o = minimal_clean(_t)
    _ok = bool(_o) and len(_o) >= max(2, len(_t.replace(' ', '')) // 3)
    if not _ok:
        _bad.append(_lang)
    print(f'    {"OK " if _ok else "FAIL"} {_lang:<9} {len(_t):>3} chars -> {len(_o):>3} chars')
if _bad:
    raise RuntimeError(f'minimal_clean is still destroying: {_bad}. '
                       'The embedding would silently lose these respondents.')
print('  all scripts preserved for the multilingual embedder.')


## Stage 14 — Apply cleaning (honours the screening flags)

Applies the cleaning functions to every question, producing the `bert_*` columns the embedding reads and the `clean_*` columns the keyword stage reads. Deterministic and recomputed each run.

In [ ]:
# @title
# ==============================================================================
# STAGE 12  -  APPLY CLEANING   (runs AFTER outlier screening)
# Produces the bert_* (light, script-agnostic) and clean_* (heavy) text columns
# that the embedding and keyword stages consume, then saves df_cleaned.
# Deterministic; recomputed each run.
# ==============================================================================
# ORDERING + MASKING FIX. Two things changed:
#  1. This stage now runs AFTER outlier screening, so `df` here is already the
#     filtered frame (participants failing all five answers are gone).
#  2. Responses individually flagged `valid_<q> == False` are NaN'd BEFORE the
#     bert_* column is built. Previously the valid_* flags were computed and then
#     never applied to anything the embedder read, so junk/copy-paste answers were
#     embedded and clustered exactly like real ones.
# ==============================================================================
import numpy as np

print('=' * 66)
print('  STAGE 12 - TEXT CLEANING  (outlier-screened input)')
print('=' * 66)

_have_flags = all(f'valid_{s}' in df.columns for s in SHORT_LABELS)
if not _have_flags:
    raise RuntimeError('valid_* flags missing - run the outlier-detection stage '
                       'BEFORE this cell (it must screen the raw text first).')

_masked_total = 0
for col, short in zip(TEXT_COLS, SHORT_LABELS):
    is_ow = (col == ONE_WORD_COL)

    # Apply the screening decision to the text the embedder will see.
    src = df[col].where(df[f'valid_{short}'], other=np.nan)
    n_masked = int(df[col].notna().sum() - src.notna().sum())
    _masked_total += n_masked

    df[f'bert_{short}']  = src.apply(minimal_clean).replace('', np.nan)
    df[f'clean_{short}'] = src.apply(
        lambda x: heavy_clean(x, is_one_word_col=is_ow)).replace('', np.nan)

    n_orig  = int(df[col].notna().sum())
    n_bert  = int(df[f'bert_{short}'].notna().sum())
    n_clean = int(df[f'clean_{short}'].notna().sum())
    print(f'  {short}')
    print(f'    raw {n_orig:>4}  |  screened out {n_masked:>3}  |  '
          f'bert_* {n_bert:>4}  |  clean_* {n_clean:>4}')

print()
print(f'  responses NaN\'d by the quality rules before embedding : {_masked_total}')
print('  (these no longer reach the embedder - previously they did)')

print()
print('  LANGUAGE DISTRIBUTION (raw text, Q1-Q4)')
for col, short in zip(TEXT_COLS[:4], SHORT_LABELS[:4]):
    langs = df[col].dropna().apply(detect_language)
    top   = Counter(langs).most_common(4)
    print(f'  {short:<12}: ' + '  '.join([f'{l}={c}' for l, c in top]))

# A non-English response must survive into bert_* now. Report it explicitly so a
# regression in minimal_clean is visible here rather than silently shrinking N.
_non_en_kept = 0
for col, short in zip(TEXT_COLS[:4], SHORT_LABELS[:4]):
    m = df[col].notna() & df[f'bert_{short}'].notna()
    _non_en_kept += int(sum(detect_language(t) != 'en' for t in df.loc[m, col]))
print(f'  non-English responses retained in bert_* : {_non_en_kept}  '
      '(0 here would mean the script filter regressed)')

save_ckpt('df_cleaned', df)
# ROUND-2 FIX (contract repair): the reorder moved outlier REMOVAL before
# cleaning, so the df_final saved at the removal stage has NO bert_* columns.
# NB2 prefers df_final (get_df()) and reads bert_* from it for c-TF-IDF terms,
# LLM quotes and valence - it would have seen empty text. df_final is therefore
# finalised HERE, after cleaning: screened + cleaned + bert_*/clean_* present.
save_ckpt('df_final', df)
print('  df_final finalised (screened + cleaned, bert_* present) - NB2 reads this.')
print('=' * 66)


## Stage 15 — Post-clean snapshot

A snapshot of the corpus after cleaning and outlier removal, to confirm the dataset that enters the analysis.

In [ ]:
# @title
# ════════════════════════════════════════════════════════════════
# STAGE 17  -  S5_POST_CLEANING
# INPUT:  df_cleaned.pkl (S2C), TEXT_COLS, SHORT_LABELS, NAT_COLORS
# OUTPUT: chart_s5_postcleaning.png
# ════════════════════════════════════════════════════════════════
# Redesigned: grouped HORIZONTAL bars (one block per question, three stage bars
# inside it). This avoids the line-chart label collisions that happen when
# several questions sit at nearly the same value, and reads left->right as a
# clear "how much was kept / how much shrank" story.
#   Panel A - valid responses per stage (Raw / BERT-input / word-cloud-Clean)
#   Panel B - mean word count per stage, with the Clean %-change called out
# BERT-input cleaning keeps almost everything; the word-cloud column is heavier
# (POS-filtered, non-English Q1-Q4 dropped) so it is shorter BY DESIGN.
#
# Always reloads df_cleaned fresh from Drive so a stale in-memory df can never
# make the cleaned columns read as zero.

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import Image, display

# ── Load the cleaned frame for this snapshot ─────────────────────
# FIX: this cell does `df = load_ckpt('df_cleaned')`, rebinding the global `df`.
# In the OLD cell order it ran AFTER outlier removal, so that rebind silently
# RESURRECTED the removed participants into the working frame for every cell
# below it (df_cleaned was the pre-removal frame). With the corrected order
# - screen -> remove -> clean -> snapshot -> embed - df_cleaned IS the screened
# frame, so this reload is now a no-op rather than a regression. We assert that
# invariant instead of trusting the ordering.
df = load_ckpt('df_cleaned')
if df is None:
    raise RuntimeError('df_cleaned.pkl not found - run the cleaning stage first.')
if not all(f'valid_{s}' in df.columns for s in SHORT_LABELS):
    raise RuntimeError('df_cleaned has no valid_* flags -> it was saved before '
                       'outlier screening. Re-run the stages in order.')
if 'to_remove' in globals() and to_remove:
    _back = set(df['Participant']) & set(to_remove)
    if _back:
        raise RuntimeError(f'df_cleaned still contains {len(_back)} removed '
                           'participant(s): the reload would resurrect them.')

# ── Verify cleaned columns exist + populated ─────────────────────
print('=' * 60)
print('  S5 - cleaned-column check on df_cleaned')
print('=' * 60)
_missing = []
for col, short in zip(TEXT_COLS, SHORT_LABELS):
    b, c = f'bert_{short}', f'clean_{short}'
    if b not in df.columns or c not in df.columns:
        _missing.append(short)
    print(f'  {short:<12} raw={int(df[col].notna().sum()):<4} '
          f'bert={int(df[b].notna().sum()) if b in df.columns else "MISSING":<5} '
          f'clean={int(df[c].notna().sum()) if c in df.columns else "MISSING"}')
if _missing:
    raise RuntimeError(f'bert_/clean_ columns missing for {_missing} - re-run S2C fresh.')

def _meanwords(series):
    s = series.dropna()
    return float(s.apply(lambda x: len(str(x).split())).mean()) if len(s) else 0.0

rows = []
for col, short, color in zip(TEXT_COLS, SHORT_LABELS, NAT_COLORS):
    b, c = f'bert_{short}', f'clean_{short}'
    rows.append(dict(
        short=short, color=color,
        n=[int(df[col].notna().sum()), int(df[b].notna().sum()), int(df[c].notna().sum())],
        w=[_meanwords(df[col]), _meanwords(df[b]), _meanwords(df[c])]))

# stage styling: Raw = light, BERT = mid, Clean = full colour (darkening = more
# processed). Shared shade ramp per question colour.
import matplotlib.colors as mcolors
def _shades(hexc):
    base = np.array(mcolors.to_rgb(hexc))
    return [tuple(base + (1 - base) * f) for f in (0.55, 0.28, 0.0)]  # light->full
stage_names = ['Raw', 'BERT-input', 'Clean (word-cloud)']

fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
fig.patch.set_facecolor(NAT_BG)
nq = len(rows)
y_base = np.arange(nq)[::-1]          # first question on top
bar_h = 0.26

# ── Panel A - response retention ─────────────────────────────────
ax = axes[0]; ax.set_facecolor('#FAFBFC')
for qi, d in enumerate(rows):
    sh = _shades(d['color'])
    for si in range(3):
        y = y_base[qi] + (1 - si) * bar_h
        ax.barh(y, d['n'][si], height=bar_h, color=sh[si],
                edgecolor='white', linewidth=0.8, zorder=3)
        ax.text(d['n'][si] + 3, y, f"{d['n'][si]}", va='center', ha='left',
                fontsize=8.5, color=d['color'], fontweight='bold')
ax.set_yticks(y_base)
ax.set_yticklabels([d['short'] for d in rows], fontsize=10.5,
                   fontweight='bold', color=NAT_TEXT)
ax.set_xlabel('Valid response count', fontsize=11, color=NAT_TEXT, labelpad=8)
ax.set_title('A  Response Retention Across Cleaning Stages',
             fontsize=12, fontweight='bold', color=NAT_TEXT, pad=12, loc='left')
ax.set_xlim(0, max(max(d['n']) for d in rows) * 1.14)
ax.xaxis.grid(True, color=NAT_GRID, linewidth=0.5, zorder=0); ax.set_axisbelow(True)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_edgecolor(NAT_GRID)
ax.tick_params(axis='x', colors='#888888', labelsize=10); ax.tick_params(axis='y', length=0)

# ── Panel B - mean word count ────────────────────────────────────
ax2 = axes[1]; ax2.set_facecolor('#FAFBFC')
for qi, d in enumerate(rows):
    sh = _shades(d['color'])
    for si in range(3):
        y = y_base[qi] + (1 - si) * bar_h
        ax2.barh(y, d['w'][si], height=bar_h, color=sh[si],
                 edgecolor='white', linewidth=0.8, zorder=3)
    # label only Raw and Clean (+pct) to keep it clean
    ax2.text(d['w'][0] + 0.3, y_base[qi] + bar_h, f"{d['w'][0]:.1f}", va='center',
             ha='left', fontsize=8.3, color=d['color'])
    pct = ((d['w'][2] - d['w'][0]) / d['w'][0] * 100) if d['w'][0] > 0 else 0.0
    ax2.text(d['w'][2] + 0.3, y_base[qi] - bar_h, f"{d['w'][2]:.1f} ({pct:+.0f}%)",
             va='center', ha='left', fontsize=8.3, color=d['color'], fontweight='bold')
ax2.set_yticks(y_base)
ax2.set_yticklabels([d['short'] for d in rows], fontsize=10.5,
                    fontweight='bold', color=NAT_TEXT)
ax2.set_xlabel('Mean word count', fontsize=11, color=NAT_TEXT, labelpad=8)
ax2.set_title('B  Word Count Reduction Across Stages',
              fontsize=12, fontweight='bold', color=NAT_TEXT, pad=12, loc='left')
ax2.set_xlim(0, max(max(d['w']) for d in rows) * 1.30)
ax2.xaxis.grid(True, color=NAT_GRID, linewidth=0.5, zorder=0); ax2.set_axisbelow(True)
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.spines['bottom'].set_edgecolor(NAT_GRID)
ax2.tick_params(axis='x', colors='#888888', labelsize=10); ax2.tick_params(axis='y', length=0)

# shared stage legend (grey ramp explains the within-question shading)
grey = ['#C9C9C9', '#9A9A9A', '#5A5A5A']
leg = [Patch(facecolor=grey[i], edgecolor='white', label=stage_names[i]) for i in range(3)]
fig.legend(handles=leg, loc='lower center', ncol=3, fontsize=9.5, frameon=False,
           bbox_to_anchor=(0.5, -0.02), title='Stage (light -> dark = more processed)',
           title_fontsize=9.5)

fig.suptitle('S5  Post-Cleaning Text Transformation Pipeline',
             fontsize=14, fontweight='bold', color=NAT_TEXT, y=1.02)
fig.text(0.5, -0.10,
         'Within each question, the three bars are Raw -> BERT-input -> word-cloud-Clean. '
         'BERT-input keeps almost every response (light cleaning); the word-cloud column '
         'is POS-filtered and drops non-English Q1–Q4, so it is shorter by design. '
         'BGE-M3 embeds all responses natively in any language.',
         ha='center', fontsize=8.4, color='#AAAAAA', style='italic')

plt.tight_layout(rect=[0, 0.02, 1, 0.99])
# round 2: routed through save_fig (was a bare 180-dpi PNG outside the manifest)
save_fig(fig, 'chart_s5_postcleaning')
plt.close(fig)

## Stage 16 — Sentence embedding (Qwen3-Embedding-8B)

Each cleaned response is encoded and L2-normalised, so similarity is cosine: $\cos(\mathbf{u},\mathbf{v})=\mathbf{u}\cdot\mathbf{v}$ for unit vectors. The model is Qwen3-Embedding-8B (4096-dimensional), a 2025 multilingual model that tops the MTEB board and is instruction-aware, so we pass a short task description framing the goal as thematic clustering. It is sized for an L4 High-RAM Colab runtime: the weights load with low CPU memory and a capped batch, and the GPU is released before the labelling model loads. The result is content-hash cached on a signature of the texts, so a re-run loads identical vectors and switching the model recomputes under a new key. Set `EMBED_BACKEND` to `qwen3` (4B, 2560-d) or `bge-m3` (1024-d) to compare. Refs: Zhang et al. (2025), Qwen3-Embedding; Chen et al. (2024), BGE-M3.

In [ ]:
# @title
# ==============================================================================
# STAGE 13  -  SENTENCE EMBEDDING  (Qwen3-Embedding by default; cached, deterministic)
# Each cleaned response is encoded and L2-normalised, so similarity is cosine.
# The default model is Qwen3-Embedding-4B (2560-d), a 2025 multilingual model that
# tops the MTEB board; it is instruction-aware, so we pass a short task description
# that frames the goal as thematic clustering (Zhang et al. 2025, Qwen3-Embedding).
# Set EMBED_BACKEND="bge-m3" in the Control Panel to fall back to the 1024-d model
# (Chen et al. 2024) for comparison. The model runs in evaluation mode with fixed
# seeds and the whole result is content-hash cached on a signature of the texts:
# a re-run loads the identical vectors, and changing the model recomputes under a
# new key. Weights live in the Drive cache (Stage 2), so they download only once.
# ==============================================================================
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# ORDERING GUARD. This stage MUST run on the outlier-screened frame. Previously
# it ran BEFORE outlier detection, so the embeddings (and therefore every theme
# downstream) were built from unfiltered text and the removal stage was inert.
# df_cleaned is now written by the post-screening cleaning stage, so loading it
# here is correct - but we assert the invariants rather than trust the order.
# ---------------------------------------------------------------------------
try:
    _ = df
    if df is None:
        df = load_ckpt("df_cleaned")
except NameError:
    df = load_ckpt("df_cleaned")
if df is None:
    raise RuntimeError("Run the cleaning stage first (need df_cleaned with bert_* columns).")

_missing_flags = [s for s in CFG.short_labels if f"valid_{s}" not in df.columns]
if _missing_flags:
    raise RuntimeError(
        "df_cleaned carries no valid_* flags -> it was built before outlier "
        "screening. Re-run: outlier detect -> outlier removal -> apply cleaning -> "
        "this cell. (Embedding unscreened text is the bug this guard exists to catch.)")

if "to_remove" in globals() and to_remove:
    _still = set(df["Participant"]) & set(to_remove)
    if _still:
        raise RuntimeError(
            f"{len(_still)} participant(s) flagged for removal are still in df. "
            "Run the outlier-removal stage before embedding.")
print(f"  ordering guard OK: embedding {len(df)} screened participants "
      f"(valid_* applied, {len(globals().get('to_remove', []) or [])} removed upstream).")

per_q = {}
for q in CFG.short_labels:
    bert_col = f"bert_{q}"
    if bert_col not in df.columns:
        raise KeyError(f"Missing cleaned column {bert_col}. Run the cleaning stage.")
    valid = df[[bert_col, "Participant", "Team"]].dropna(subset=[bert_col])
    per_q[q] = (valid[bert_col].astype(str).tolist(),
                valid[["Participant", "Team"]].reset_index(drop=True))

all_texts = [t for q in CFG.short_labels for t in per_q[q][0]]
_use_instruction = CFG.embed_backend.startswith("qwen3")
_instruction = globals().get("EMBED_INSTRUCTION", "")
# The 8B model nearly fills an L4 (22.5 GB); cap the batch so encoding does not OOM.
_BS = min(CFG.embed_batch, 8) if CFG.embed_backend == "qwen3-8b" else CFG.embed_batch

def _load_model():
    has_cuda = torch.cuda.is_available()
    if CFG.embed_backend.startswith("qwen3"):
        kw = dict(trust_remote_code=True,
                  model_kwargs={"torch_dtype": torch.float16, "low_cpu_mem_usage": True},
                  tokenizer_kwargs={"padding_side": "left"})
        if has_cuda:
            kw["device"] = "cuda"
        m = SentenceTransformer(CFG.embed_model, **kw)
    else:
        m = SentenceTransformer(CFG.embed_model, device="cuda" if has_cuda else None)
    try:
        m.eval()
    except Exception:
        pass
    return m

def _encode(model, texts):
    # Qwen3 uses an instruction; older sentence-transformers lack a prompt= kwarg,
    # so fall back to prepending the instruction manually.
    if _use_instruction and _instruction:
        try:
            return model.encode(texts, batch_size=_BS, prompt=_instruction,
                                 normalize_embeddings=True, show_progress_bar=True)
        except TypeError:
            texts = [f"{_instruction} {t}" for t in texts]
    return model.encode(texts, batch_size=_BS,
                        normalize_embeddings=True, show_progress_bar=True)

def _compute_embeddings():
    set_all_seeds()
    print(f"  loading {CFG.embed_model} (first run downloads weights to the Drive cache) ...")
    model = _load_model()
    ed, md = {}, {}
    for q in CFG.short_labels:
        texts, meta = per_q[q]
        ed[q] = np.asarray(_encode(model, texts), dtype=np.float32)
        md[q] = meta
    # free the GPU so the labelling LLM (Stage 23) has room
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"embeddings": ed, "metadata": md}

_key = CFG.fingerprint("embed", len(all_texts), extra=data_sig(all_texts))
_bundle = cached("embed", _key, _compute_embeddings)
embeddings_dict = _bundle["embeddings"]
metadata_dict = _bundle["metadata"]

print("=" * 64)
print(f"  STAGE 13  EMBEDDING  ({CFG.embed_model}, {CFG.embed_dim}-d, L2-normalised)")
print("=" * 64)
if _use_instruction:
    print(f"  instruction: {_instruction[:70]}...")
for q in CFG.short_labels:
    e = embeddings_dict[q]
    print(f"    {q:<13} shape={str(e.shape):<13} norm mean={np.linalg.norm(e, axis=1).mean():.4f}")
print("=" * 64)


## Stage 16b — Persist embedding artifacts
Save `embeddings_dict` / `metadata_dict` under explicit names so a fresh kernel (or Notebook 2) loads them without re-embedding.

In [ ]:
# @title
# ==============================================================================
# STAGE 13b  -  PERSIST EMBEDDING ARTIFACTS  (explicit, restart-safe)
# The monolith kept embeddings_dict / metadata_dict only in memory (plus the
# content-hash cache), so a fresh kernel that jumped straight to pooling could not
# find them. Here we persist them under explicit names, so Notebook 2 (and any
# re-run) can load them directly. These two dicts are usually well under the size
# cap; the big stacked matrix is saved by the pooling stage as pooled_embeddings.
# ==============================================================================
try:
    _ = embeddings_dict
    _ = metadata_dict
except NameError:
    raise RuntimeError("Run the embedding stage first (need embeddings_dict + metadata_dict).")

psave("embeddings_dict", embeddings_dict)
psave("metadata_dict", metadata_dict)
print("  persisted: embeddings_dict, metadata_dict  ->", POOLED_DIR)
for q in CFG.short_labels:
    print(f"    {q:<13} {np.asarray(embeddings_dict[q]).shape}")


## Stage 17 — Embedding quality

Validates the embeddings before any projection with metrics the field uses: an L2-norm sanity check, the corpus anisotropy baseline (instruction-tuned encoders are anisotropic, so cosines are read relative to it), a question-by-question similarity matrix with self-pairs excluded and centered on that baseline, a within-minus-between separation per question with bootstrap CIs, and the effective dimensionality (participation ratio, PCA variance). Small separation is expected because pooled themes are designed to cross questions. Refs: Ethayarajh 2019; Mu and Viswanath 2018; Reimers and Gurevych 2019; Wang and Isola 2020.

In [ ]:
# @title
# ==============================================================================
# STAGE 14  -  EMBEDDING QUALITY  (literature-grounded validation)
# Validates the embeddings before any projection, with metrics the NLP community
# actually uses, not a raw cosine matrix (which is misleading for instruction
# models). Four panels:
#   A  L2-norm distribution            -> confirms cosine geometry (norms approx 1)
#   B  question x question similarity, SELF-PAIRS EXCLUDED and CENTERED on the
#      corpus anisotropy baseline       -> the meaningful signal above chance
#   C  within- minus between-question separation, per question, bootstrap 95% CI
#      -> are responses to a question internally cohesive vs the rest of the corpus
#   D  effective dimensionality (PCA cumulative variance + participation ratio)
#      -> how isotropic / how many dimensions the embedding really uses
# The anisotropy baseline (mean cosine of random pairs) is reported because
# instruction-tuned encoders are highly anisotropic; raw cosines look high and
# compressed, so within and between values must be read RELATIVE to it.
# Refs: Ethayarajh (2019); Mu and Viswanath (2018); Reimers and Gurevych (2019);
#       Wang and Isola (2020, alignment/uniformity).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    _ = embeddings_dict
except NameError:
    embeddings_dict = load_ckpt("embeddings_dict")
if embeddings_dict is None:
    raise RuntimeError("Run the embedding stage first (need embeddings_dict).")

QL = list(SHORT_LABELS)
set_all_seeds()
rng = np.random.default_rng(CFG.seed)

emb_all = np.vstack([np.asarray(embeddings_dict[q]) for q in QL]).astype(np.float32)
n_all = len(emb_all)

# corpus anisotropy baseline: mean cosine of random distinct pairs (Ethayarajh 2019)
si = rng.choice(n_all, size=min(2000, n_all), replace=False)
Ssamp = emb_all[si] @ emb_all[si].T
np.fill_diagonal(Ssamp, np.nan)
aniso = float(np.nanmean(Ssamp))

# self-similarity sanity (should be ~1.0 for L2-normalised vectors)
self_sim = float(np.mean([(np.asarray(embeddings_dict[q]) *
                           np.asarray(embeddings_dict[q])).sum(1).mean() for q in QL]))

# within (excl self) and between, per question
within, between, within_se = {}, {}, {}
sim_mat = np.zeros((len(QL), len(QL)))
for i, q in enumerate(QL):
    e = np.asarray(embeddings_dict[q])
    W = e @ e.T; np.fill_diagonal(W, np.nan)
    within[q] = float(np.nanmean(W))
    boot = []
    for _ in range(150):
        b = rng.choice(len(e), len(e), replace=True)
        Wb = e[b] @ e[b].T
        # ROUND-2 FIX: sampling WITH replacement duplicates rows, so a row's
        # self-pair (cosine = 1.0) lands OFF the diagonal; fill_diagonal missed
        # those and inflated the bootstrap within-question mean. Mask every pair
        # drawn from the same original row. Same RNG draws -> same seed path.
        Wb[b[:, None] == b[None, :]] = np.nan
        boot.append(np.nanmean(Wb))
    within_se[q] = float(np.std(boot))
    others = np.vstack([np.asarray(embeddings_dict[o]) for o in QL if o != q])
    between[q] = float((e @ others.T).mean())
    for j, q2 in enumerate(QL):
        if i == j:
            sim_mat[i, j] = within[q]
        else:
            e2 = np.asarray(embeddings_dict[q2])
            sim_mat[i, j] = float((e @ e2.T).mean())
centered = sim_mat - aniso          # signal above the anisotropy baseline

# effective dimensionality: participation ratio of the spectrum
Xc = emb_all - emb_all.mean(0)
sv = np.linalg.svd(Xc, compute_uv=False)
lam = sv ** 2
part_ratio = float((lam.sum() ** 2) / (lam ** 2).sum())   # ~1 anisotropic, ~D isotropic
cum = np.cumsum(lam) / lam.sum()
d90 = int(np.searchsorted(cum, 0.90) + 1)

print("=" * 64)
print(f"  STAGE 14  EMBEDDING QUALITY  ({CFG.embed_model}, {CFG.embed_dim}-d)")
print("=" * 64)
print(f"  self-similarity        : {self_sim:.4f}   (target 1.000; sanity of L2 norm)")
print(f"  anisotropy baseline    : {aniso:.3f}   (mean cosine of random pairs; high = anisotropic)")
print(f"  effective dimensions   : participation ratio {part_ratio:.1f}; {d90} dims hold 90% variance")
print("  within vs between (centered on baseline):")
for q in QL:
    sep = within[q] - between[q]
    print(f"    {q:<13} within {within[q]-aniso:+.3f}  between {between[q]-aniso:+.3f}  separation {sep:+.3f}")
print("  reading: small positive separation is expected here; pooled themes are meant to")
print("           cross questions, so questions overlap in embedding space by design.")

NAT_BG_ = globals().get("NAT_BG", "#FFFFFF")
fig = plt.figure(figsize=(17, 9)); fig.patch.set_facecolor(NAT_BG_)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34,
                       left=0.07, right=0.95, top=0.88, bottom=0.10)

# A cosine similarity distribution: within vs between vs random (anisotropy + signal)
axA = fig.add_subplot(gs[0, 0])
_rngA = np.random.default_rng(CFG.seed + 7)
_origin_all = np.concatenate([[qi] * len(np.asarray(embeddings_dict[QL[qi]])) for qi in range(len(QL))])
_M = 9000
_ri = _rngA.integers(0, n_all, _M); _rj = _rngA.integers(0, n_all, _M)
_cos = np.sum(emb_all[_ri] * emb_all[_rj], axis=1)
_ok = _ri != _rj
_win = _cos[(_origin_all[_ri] == _origin_all[_rj]) & _ok]
_bet = _cos[(_origin_all[_ri] != _origin_all[_rj]) & _ok]
_rnd = _cos[_ok]
_bins = np.linspace(min(_rnd.min(), _bet.min()) - 0.02, 1.0, 50)
axA.hist(_rnd, bins=_bins, color="#C9C9C9", alpha=0.65, density=True, label=f"random pairs (mean {_rnd.mean():.2f})")
axA.hist(_bet, bins=_bins, color="#6699CC", alpha=0.45, density=True, label=f"between-question (mean {_bet.mean():.2f})")
axA.hist(_win, bins=_bins, color="#882255", alpha=0.45, density=True, label=f"within-question (mean {_win.mean():.2f})")
for _v, _c in [(_rnd.mean(), "#888"), (_bet.mean(), "#6699CC"), (_win.mean(), "#882255")]:
    axA.axvline(_v, color=_c, ls="--", lw=1.2)
axA.set_title("A  Cosine similarity distribution (within > between > baseline)", fontsize=10.5, fontweight="bold", loc="left")
axA.set_xlabel("cosine similarity"); axA.set_yticks([])
axA.legend(fontsize=7.5, frameon=False, loc="upper left")
axA.text(0.98, 0.96, f"L2 norm = {self_sim:.3f}\n(cosine geometry OK)", transform=axA.transAxes,
         fontsize=7.5, va="top", ha="right", color="#888")

# B centered similarity matrix
from mpl_toolkits.axes_grid1 import make_axes_locatable
axB = fig.add_subplot(gs[0, 1])
vmax = np.abs(centered).max() or 1e-6
im = axB.imshow(centered, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
for i in range(len(QL)):
    for j in range(len(QL)):
        axB.text(j, i, f"{sim_mat[i, j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if abs(centered[i, j]) > 0.6 * vmax else "#333")
axB.set_xticks(range(len(QL))); axB.set_yticks(range(len(QL)))
axB.set_xticklabels(QL, rotation=20, ha="right", fontsize=8); axB.set_yticklabels(QL, fontsize=8)
axB.set_title(f"B  Cosine similarity (text = raw cosine; colour = vs baseline {aniso:.2f})\n"
              "diagonal = within-question, self-pairs excluded", fontsize=10.0, fontweight="bold", loc="left")
_div = make_axes_locatable(axB); _cax = _div.append_axes("right", size="5%", pad=0.08)
plt.colorbar(im, cax=_cax)

# C within - between separation with CI
axC = fig.add_subplot(gs[1, 0])
sep = np.array([within[q] - between[q] for q in QL])
se = np.array([within_se[q] for q in QL])
xs = np.arange(len(QL))
axC.bar(xs, sep, color=palette(len(QL)), alpha=0.85, zorder=3)
axC.errorbar(xs, sep, yerr=1.96 * se, fmt="none", ecolor="#555", capsize=4, zorder=4)
axC.axhline(0, color="#999", lw=1)
axC.set_xticks(xs); axC.set_xticklabels(QL, rotation=15, ha="right", fontsize=8)
axC.set_ylabel("within - between cosine")
axC.set_title("C  Question cohesion vs separation (95% CI)\n"
              "near 0 = overlaps other questions (expected for pooling)",
              fontsize=10.5, fontweight="bold", loc="left")

# D effective dimensionality
axD = fig.add_subplot(gs[1, 1])
axD.plot(np.arange(1, len(cum) + 1), cum, color=HIGHLIGHT, lw=2)
axD.axhline(0.90, color="#999", ls="--", lw=1)
axD.axvline(d90, color="#999", ls=":", lw=1)
axD.set_xlim(1, min(200, len(cum)))
axD.set_xlabel("PCA components"); axD.set_ylabel("cumulative variance")
axD.set_title(f"D  Effective dimensionality\n participation ratio {part_ratio:.0f}; "
              f"{d90} dims = 90% variance", fontsize=10.5, fontweight="bold", loc="left")

fig.suptitle(f"Stage 9  Embedding quality   {CFG.embed_model}  ·  {CFG.embed_dim}-d  ·  L2-normalised",
             fontsize=13.5, fontweight="bold", y=0.96)
fig.text(0.5, 0.02, "Anisotropy baseline + within/between separation + effective dimensionality. "
         "Refs: Ethayarajh 2019; Mu & Viswanath 2018; Reimers & Gurevych 2019; Wang & Isola 2020.",
         ha="center", fontsize=8, style="italic", color="#888")
save_fig(fig, "chart_s3b_embedding_validation")
plt.close(fig)

# (interactive twin removed: this validation cell is static-only)
psave("embedding_quality", dict(anisotropy=aniso, self_sim=self_sim, participation_ratio=part_ratio,
                                d90=d90, within=within, between=between, sim_matrix=sim_mat))
print("=" * 64)


## Stage 14b — Language composition & multilingual retention (new)

The pipeline rests on a multilingual claim (Lingua identifies 74 languages; Qwen3-Embedding encodes them natively) but never showed the language mix, and never checked that non-English text survived cleaning — it did not. Panel A reports the mix; Panel B is a regression guard: every script must retain ~100% into the embedder input. A short bar in B means the script filter is dropping respondents again.


In [ ]:
# @title
# ==============================================================================
# STAGE 14b  -  LANGUAGE COMPOSITION & MULTILINGUAL RETENTION   (NEW)
# INPUT:  df (screened + cleaned), TEXT_COLS, SHORT_LABELS
# OUTPUT: language_audit artifact, chart_s14b_language
# ==============================================================================
# MISSING FIGURE, added by the audit. The pipeline rests on a multilingual claim
# - Lingua detects 74 languages, Qwen3-Embedding/BGE-M3 encode them natively, and
# the methods text says "embeds all responses natively in any language" - yet
# nothing anywhere VISUALISED the language mix or checked that non-English text
# actually survived cleaning. It did not: the old minimal_clean kept only Latin
# and CJK characters, so Cyrillic / Devanagari / Hangul / Greek / Arabic answers
# were silently reduced to '' -> NaN -> dropped before the embedder ever saw them,
# and Japanese was cut down to its kanji. With no figure and no retention check,
# an entire class of respondents could vanish and every printed N would still look
# perfectly healthy.
#
# This figure is therefore both a descriptive panel AND a regression guard:
#   A  language mix of the raw corpus (which languages, how many responses)
#   B  retention THROUGH cleaning per script family: raw -> bert_* (embedder input)
#      A bar short of 100% in B means minimal_clean is dropping that script again.
# Panel B is the check that would have caught the original bug on sight.
# Refs: Lingua (language ID); Zhang et al. (2025) Qwen3-Embedding (multilingual).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from collections import Counter
import unicodedata

if "df" not in globals() or df is None:
    df = load_ckpt("df_cleaned")
if df is None:
    raise RuntimeError("Run the cleaning stage first (need df_cleaned with bert_* columns).")

# ---- script family of a string (by the modal Unicode block of its letters) ----
def _script_of(s):
    names = []
    for ch in str(s):
        if not ch.isalpha():
            continue
        try:
            n = unicodedata.name(ch)
        except ValueError:
            continue
        for tag in ("LATIN", "CYRILLIC", "GREEK", "ARABIC", "HEBREW", "DEVANAGARI",
                    "HANGUL", "HIRAGANA", "KATAKANA", "CJK", "THAI", "BENGALI", "TAMIL"):
            if n.startswith(tag) or f" {tag} " in n:
                names.append("Japanese kana" if tag in ("HIRAGANA", "KATAKANA")
                             else ("CJK / Chinese" if tag == "CJK" else tag.title()))
                break
    return Counter(names).most_common(1)[0][0] if names else "Other/None"

# ---- gather per-response language + script, raw vs surviving into bert_* ------
rows = []
for col, short in zip(TEXT_COLS, SHORT_LABELS):
    raw = df[col]
    bert = df[f"bert_{short}"]
    for idx in df.index:
        v = raw.loc[idx]
        if pd.isna(v) or not str(v).strip():
            continue
        rows.append(dict(question=short,
                         lang=detect_language(v),
                         script=_script_of(v),
                         kept=bool(pd.notna(bert.loc[idx]) and str(bert.loc[idx]).strip())))
L = pd.DataFrame(rows)
if L.empty:
    raise RuntimeError("No non-empty responses found - check the data-load stage.")

lang_counts = L["lang"].value_counts()
by_script = (L.groupby("script")
               .agg(n_raw=("kept", "size"), n_kept=("kept", "sum"))
               .assign(retention=lambda d: d["n_kept"] / d["n_raw"])
               .sort_values("n_raw", ascending=False))

n_non_en = int((L["lang"] != "en").sum())
n_non_en_kept = int(((L["lang"] != "en") & L["kept"]).sum())
overall_ret = float(L["kept"].mean())

psave("language_audit", dict(lang_counts=lang_counts.to_dict(),
                             by_script=by_script.to_dict(),
                             n_non_en=n_non_en, n_non_en_kept=n_non_en_kept,
                             overall_retention=overall_ret))

print("=" * 66)
print("  STAGE 14b  LANGUAGE COMPOSITION & MULTILINGUAL RETENTION")
print("=" * 66)
print(f"  responses (non-blank)      : {len(L)}")
print(f"  languages detected         : {L['lang'].nunique()}")
print(f"  non-English responses      : {n_non_en}  ({100*n_non_en/len(L):.1f}%)")
print(f"  non-English kept in bert_* : {n_non_en_kept} / {n_non_en}"
      + (f"  ({100*n_non_en_kept/max(n_non_en,1):.1f}%)" if n_non_en else ""))
print()
print(f"  {'script':<16}{'raw':>7}{'kept':>7}{'retention':>11}")
print("  " + "-" * 41)
for sc, r in by_script.iterrows():
    flag = "   <-- DROPPING" if r["retention"] < 0.95 else ""
    print(f"  {sc:<16}{int(r['n_raw']):>7}{int(r['n_kept']):>7}{r['retention']:>10.1%}{flag}")

# regression guard: any script losing >5% to cleaning is the old bug returning
_bad = by_script[(by_script["retention"] < 0.95) & (by_script["n_raw"] >= 3)]
if len(_bad):
    print()
    print("  *** WARNING: a script is being lost during cleaning. This is exactly the")
    print("      failure mode the Unicode-category fix removed. Inspect minimal_clean. ***")

# ------------------------------- figure --------------------------------------
_cols = palette(max(len(lang_counts), 3))
fig, ax = plt.subplots(1, 2, figsize=(15.5, max(5.2, 0.42 * len(by_script) + 3.4)),
                       facecolor=NAT_BG, gridspec_kw=dict(width_ratios=[1.25, 1.0], wspace=0.34))

# A  language mix
top = lang_counts.head(12)[::-1]
ax[0].barh(range(len(top)), top.values,
           color=[_cols[i % len(_cols)] for i in range(len(top))],
           edgecolor="white", height=0.72, zorder=3)
for i, v in enumerate(top.values):
    ax[0].text(v + max(top.values) * 0.012, i, f"{v}  ({100*v/len(L):.1f}%)",
               va="center", fontsize=9, color=NAT_TEXT)
ax[0].set_yticks(range(len(top)))
ax[0].set_yticklabels([str(x).upper() for x in top.index], fontsize=10)
ax[0].set_xlim(0, max(top.values) * 1.22)
ax[0].set_xlabel("responses")
ax[0].set_title(f"A   Language mix of the corpus  ({L['lang'].nunique()} languages, "
                f"{100*n_non_en/len(L):.0f}% non-English)",
                fontsize=12, fontweight="bold", loc="left", color=NAT_TEXT, pad=10)
ax[0].xaxis.grid(True, color="#F0F0F0", zorder=0); ax[0].set_axisbelow(True)

# B  retention through cleaning, per script
sc = by_script[::-1]
ok = "#44AA99"; bad = "#CC6677"
bars = ax[1].barh(range(len(sc)), sc["retention"].values * 100,
                  color=[ok if r >= 0.95 else bad for r in sc["retention"]],
                  edgecolor="white", height=0.68, zorder=3)
ax[1].axvline(100, color="#999", ls="--", lw=1.2, zorder=4)
for i, (name, r) in enumerate(sc.iterrows()):
    ax[1].text(min(r["retention"] * 100, 100) + 1.4, i,
               f"{r['retention']:.0%}  ({int(r['n_kept'])}/{int(r['n_raw'])})",
               va="center", fontsize=9,
               color=NAT_TEXT if r["retention"] >= 0.95 else bad,
               fontweight="normal" if r["retention"] >= 0.95 else "bold")
ax[1].set_yticks(range(len(sc)))
ax[1].set_yticklabels(sc.index, fontsize=10)
ax[1].set_xlim(0, 128); ax[1].set_xlabel("% of raw responses surviving into the embedder input")
ax[1].set_title("B   Retention through cleaning, by script",
                fontsize=12, fontweight="bold", loc="left", color=NAT_TEXT, pad=10)
ax[1].legend(handles=[Patch(color=ok, label="retained (≥ 95%)"),
                      Patch(color=bad, label="being dropped (< 95%)")],
             fontsize=8.5, loc="lower right", frameon=False)
ax[1].xaxis.grid(True, color="#F0F0F0", zorder=0); ax[1].set_axisbelow(True)

for a in ax:
    a.set_facecolor("white")
    a.tick_params(length=0)
    for sp in ("top", "right", "left"):
        a.spines[sp].set_visible(False)
    a.spines["bottom"].set_color(NAT_GRID)

fig.suptitle("S14b — Language composition and multilingual retention",
             fontsize=14, fontweight="bold", color=NAT_TEXT, y=1.0)
fig.text(0.5, -0.02,
         "Panel B is a regression guard: every script must sit at ~100%. The earlier "
         "minimal_clean kept only Latin and CJK characters, so Cyrillic, Devanagari, "
         "Hangul, Greek and Arabic responses were emptied to NaN and dropped before "
         "embedding (Japanese survived only as its kanji). Nothing in the pipeline "
         "reported that loss. Language ID: Lingua; embedder: Qwen3-Embedding (multilingual).",
         ha="center", fontsize=8.3, color="#999", style="italic", wrap=True)
plt.tight_layout()
save_fig(fig, "chart_s14b_language")
plt.close(fig)
print("=" * 66)


## Stage 18 — Build the thematic pool (narrative questions, substantive answers)

The four narrative questions are stacked into one space so a cross-cutting theme can emerge whichever question raised it. Two evidence-based rules decide what enters:

1. **Instrument rule** — the *One Word* item asks for a summary label, not a narrative (median 1 word vs 11–20 for the others; 68% of its answers are ≤2 words). Clustering it with paragraphs produced length-artifact clusters, so it is analysed separately by the right method.
2. **Substance rule** — narrative answers shorter than `MIN_POOL_WORDS` cannot support a theme and were landing in the same artifact clusters.

**No response is discarded.** Excluded rows keep their embeddings in `pooled_held_*` and are assigned a theme post-hoc in Notebook 2 with an explicit confidence margin; the One-Word item additionally gets its own lexical analysis in Notebook 3. Set the two knobs to `()` and `0` to restore the previous behaviour exactly.


In [ ]:
# @title
# ==============================================================================
# STAGE 18  -  BUILD THE THEMATIC POOL  (narrative questions, substantive answers)
# ------------------------------------------------------------------------------
# WHAT CHANGED AND WHY (measured, not assumed)
# The previous version stacked ALL five questions into one thematic space. An
# audit of the resulting clusters showed that this produced a class of clusters
# that are LENGTH ARTIFACTS, not themes:
#   * word count per question:  Connection 18 · Team Work 20 · Conditions 11 ·
#     Beyond iGEM 16  BUT  One Word = 1 (median), 68% of its answers <= 2 words.
#   * 11 of 54 sub-themes had median length <= 2 words; every one of them drew
#     52-100% of its members from the One-Word item.
#   * Inspected directly, one such cluster ("Little Family Involvement", n=64,
#     median 1 word) contained: "eating meals and getting boba", "Mostly on the
#     internet", "Nothing", "We are supported by MGM Macau", "I don't know".
#     They share no topic - only their SHORTNESS. A one-word answer's embedding
#     carries almost no topical signal, so short answers sit together because
#     they are short, and Ward then carves that low-information region into
#     "themes". c-TF-IDF on ~1-word documents then makes any word occurring 2-3
#     times look highly distinctive, and the labelling model turns that into a
#     confident but unsupported theme name.
#
# THE FIX (two independent rules, both evidence-backed):
#   1. INSTRUMENT rule  - the One-Word item asks for a summary LABEL, not a
#      narrative. It is a different instrument and is analysed with the right
#      method (lexical / word-frequency analysis + alignment to the main themes),
#      not by thematic clustering.
#   2. SUBSTANCE rule   - within the narrative questions, answers below
#      MIN_POOL_WORDS tokens cannot support a theme (measured: 89 such rows,
#      7.3%, and they were landing in the same artifact clusters).
#
# NOTHING IS DISCARDED. Excluded rows are saved as a separate held-out set with
# their embeddings and the reason for exclusion, so Notebook 2 can assign them a
# theme post-hoc (with an explicit confidence margin) and Notebook 3 can run the
# one-word lexical analysis. Set the knobs below to () and 0 to restore the old
# behaviour exactly.
# ==============================================================================
import numpy as np
import pandas as pd

# ------------------------------- EDIT HERE ------------------------------------
POOL_EXCLUDE_QUESTIONS = ()             # () = pool EVERY question, One Word included
MIN_POOL_WORDS         = 0               # 0 = no length rule (old behaviour)
# ----------------------------- END EDIT ---------------------------------------

try: _ = embeddings_dict
except NameError: embeddings_dict = load_ckpt("embeddings_dict")
try: _ = metadata_dict
except NameError: metadata_dict = load_ckpt("metadata_dict")
if embeddings_dict is None or metadata_dict is None:
    raise RuntimeError("Run the embedding stage first (need embeddings_dict + metadata_dict).")

# The word count must be taken on the TEXT THE EMBEDDER ACTUALLY SAW (bert_*),
# so the rule and the geometry refer to the same object.
_df_txt = None
for _nm in ("df_final", "df_cleaned"):
    _cand = load_ckpt(_nm)
    if _cand is not None:
        _df_txt = _cand; _src_frame = _nm; break
if _df_txt is None:
    raise RuntimeError("Need df_final or df_cleaned (for the bert_* text) to apply "
                       "the substance rule. Run the cleaning stage first, or set "
                       "MIN_POOL_WORDS = 0 to skip the length rule entirely.")
_missing_cols = [f"bert_{q}" for q in CFG.short_labels if f"bert_{q}" not in _df_txt.columns]
if _missing_cols:
    raise RuntimeError(f"{_src_frame} is missing {_missing_cols}. Re-run the cleaning stage.")

_bad_q = [q for q in POOL_EXCLUDE_QUESTIONS if q not in CFG.short_labels]
if _bad_q:
    raise RuntimeError(f"POOL_EXCLUDE_QUESTIONS names unknown question(s) {_bad_q}. "
                       f"Valid: {list(CFG.short_labels)}")

# ---------------- split every question's rows into CORE and HELD --------------
core_X, core_origin, core_meta = [], [], []
held_X, held_origin, held_meta = [], [], []
report = []

for q in CFG.short_labels:
    e = np.asarray(embeddings_dict[q])
    m = metadata_dict[q].copy().reset_index(drop=True)
    if len(m) != len(e):
        raise RuntimeError(f"metadata_dict[{q!r}] has {len(m)} rows but "
                           f"embeddings_dict[{q!r}] has {len(e)} - re-run the embedding stage.")
    m["question"] = q

    # word count via Participant -> bert_<q>  (Participant is unique per row in df)
    _txt_map = dict(zip(_df_txt["Participant"], _df_txt[f"bert_{q}"]))
    _n_words = np.array([len(str(_txt_map.get(p, "")).split())
                         if pd.notna(_txt_map.get(p, np.nan)) else 0
                         for p in m["Participant"]], dtype=int)
    m["n_words"] = _n_words
    _unmatched = int(sum(1 for p in m["Participant"] if p not in _txt_map))

    is_item_excluded = q in POOL_EXCLUDE_QUESTIONS
    too_short = _n_words < int(MIN_POOL_WORDS)
    keep = np.ones(len(m), dtype=bool) if not is_item_excluded else np.zeros(len(m), dtype=bool)
    if not is_item_excluded:
        keep &= ~too_short

    reason = np.where(is_item_excluded, "non_narrative_item",
                      np.where(too_short, "below_min_words", ""))
    core_X.append(e[keep]); core_origin.append(np.array([q] * int(keep.sum())))
    core_meta.append(m[keep].reset_index(drop=True))
    hm = m[~keep].copy().reset_index(drop=True)
    hm["exclusion_reason"] = reason[~keep]
    held_X.append(e[~keep]); held_origin.append(np.array([q] * int((~keep).sum())))
    held_meta.append(hm)

    report.append(dict(question=q, total=len(m), core=int(keep.sum()),
                       held_item=int(len(m)) if is_item_excluded else 0,
                       held_short=0 if is_item_excluded else int(too_short.sum()),
                       median_words=float(np.median(_n_words)) if len(_n_words) else 0.0,
                       unmatched=_unmatched))

X_pool      = np.vstack([a for a in core_X if len(a)]).astype(np.float32)
origin_pool = np.concatenate([a for a in core_origin if len(a)])
meta_pool   = pd.concat([d for d in core_meta if len(d)], ignore_index=True)

_hX = [a for a in held_X if len(a)]
X_held      = np.vstack(_hX).astype(np.float32) if _hX else np.zeros((0, X_pool.shape[1]), np.float32)
origin_held = np.concatenate([a for a in held_origin if len(a)]) if _hX else np.array([], dtype=object)
meta_held   = (pd.concat([d for d in held_meta if len(d)], ignore_index=True)
               if any(len(d) for d in held_meta) else meta_pool.iloc[0:0].copy())

# ------------------------------ hard guards -----------------------------------
_n_all = sum(len(np.asarray(embeddings_dict[q])) for q in CFG.short_labels)
assert len(X_pool) == len(origin_pool) == len(meta_pool), "core length mismatch"
assert len(X_held) == len(origin_held) == len(meta_held), "held length mismatch"
assert len(X_pool) + len(X_held) == _n_all, (
    f"rows lost: core {len(X_pool)} + held {len(X_held)} != embedded {_n_all}")
assert len(X_pool) > 0, "the pool is empty - relax POOL_EXCLUDE_QUESTIONS / MIN_POOL_WORDS"
if MIN_POOL_WORDS > 0:
    assert int(meta_pool["n_words"].min()) >= int(MIN_POOL_WORDS), "a short row leaked into the pool"
for q in POOL_EXCLUDE_QUESTIONS:
    assert not (origin_pool == q).any(), f"{q} leaked into the pool"

policy = dict(exclude_questions=list(POOL_EXCLUDE_QUESTIONS), min_pool_words=int(MIN_POOL_WORDS),
              n_embedded=int(_n_all), n_core=int(len(X_pool)), n_held=int(len(X_held)),
              per_question=report, text_source=_src_frame,
              rationale=("One-Word is a summary-label instrument, not a narrative; short answers "
                         "cannot support a theme. Both rules were set from measured evidence "
                         "(see the stage header). Held-out rows keep their embeddings and are "
                         "assigned a theme post-hoc in NB2 with a confidence margin."))

psave("pooled_embeddings", X_pool); psave("pooled_origin", origin_pool); psave("pooled_meta", meta_pool)
psave("pooled_held_embeddings", X_held); psave("pooled_held_origin", origin_held)
psave("pooled_held_meta", meta_held); psave("pooled_pool_policy", policy)

# --------------------------------- report -------------------------------------
total = len(X_pool)
print("=" * 78)
print("  STAGE 18  THEMATIC POOL  (narrative questions, substantive answers)")
print("=" * 78)
print(f"  rule 1 (instrument) : exclude {list(POOL_EXCLUDE_QUESTIONS) or 'nothing'}")
print(f"  rule 2 (substance)  : keep answers with >= {MIN_POOL_WORDS} words"
      f"{' (disabled)' if MIN_POOL_WORDS == 0 else ''}")
print(f"  word counts taken on the embedded text (bert_*) from {_src_frame}")
print()
print(f"  {'question':<14}{'embedded':>9}{'-> pool':>9}{'held:item':>11}{'held:short':>11}{'medWords':>10}")
print("  " + "-" * 64)
for r in report:
    print(f"  {r['question']:<14}{r['total']:>9}{r['core']:>9}{r['held_item']:>11}"
          f"{r['held_short']:>11}{r['median_words']:>10.0f}")
print("  " + "-" * 64)
print(f"  {'TOTAL':<14}{_n_all:>9}{total:>9}"
      f"{sum(r['held_item'] for r in report):>11}{sum(r['held_short'] for r in report):>11}")
_unm = sum(r["unmatched"] for r in report)
if _unm:
    print(f"  !! {_unm} row(s) had no text match in {_src_frame} and were treated as 0 words.")
print()
print(f"  thematic pool : {X_pool.shape}  ({CFG.embed_dim}-d, {CFG.embed_model})")
print(f"  held out      : {X_held.shape}  -> pooled_held_* (NB2 assigns these post-hoc)")
print(f"  L2 norm mean  : {np.linalg.norm(X_pool, axis=1).mean():.4f}  (target 1.0)")
print(f"  pool word count: median {meta_pool['n_words'].median():.0f}, "
      f"min {meta_pool['n_words'].min()}, max {meta_pool['n_words'].max()}")
print("  NOTE: no response is discarded - held-out rows keep their embeddings and")
print("        receive a theme post-hoc; the One-Word item gets its own lexical analysis.")

# ------------------------------- preview figure -------------------------------
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
set_all_seeds()
_pca = PCA(n_components=2, random_state=CFG.seed); pca2 = _pca.fit_transform(X_pool)
v0, v1 = _pca.explained_variance_ratio_[:2]
NB = globals().get("NAT_BG", "#FFFFFF"); NT = globals().get("NAT_TEXT", "#1E1E1E")
pooled_qs = [q for q in CFG.short_labels if q not in POOL_EXCLUDE_QUESTIONS]

fig, ax = plt.subplots(1, 2, figsize=(14, 5), facecolor=NB,
                       gridspec_kw=dict(width_ratios=[1.0, 1.5], wspace=0.24))
rep = {r["question"]: r for r in report}
ys = list(CFG.short_labels)[::-1]
for i, q in enumerate(ys):
    r = rep[q]
    ax[0].barh(i, r["core"], color=Q_COLORS[q], edgecolor="white", height=0.66, zorder=3)
    ax[0].barh(i, r["held_short"], left=r["core"], color="#C4CAD0", edgecolor="white",
               height=0.66, zorder=3)
    ax[0].barh(i, r["held_item"], left=r["core"] + r["held_short"], color="#8A9098",
               edgecolor="white", height=0.66, zorder=3)
    ax[0].text(r["total"] + max(x["total"] for x in report) * 0.02, i,
               f"{r['core']} pooled", va="center", fontsize=8.5, color=NT)
ax[0].set_yticks(range(len(ys))); ax[0].set_yticklabels(ys, fontsize=9.5)
ax[0].set_xlim(0, max(x["total"] for x in report) * 1.30)
ax[0].set_xlabel("responses")
ax[0].set_title("A  What enters the thematic pool", fontsize=11, fontweight="bold",
                loc="left", color=NT)
import matplotlib.patches as _mp
ax[0].legend(handles=[_mp.Patch(color="#7F8A93", label="pooled (question colour)"),
                      _mp.Patch(color="#C4CAD0", label=f"held: < {MIN_POOL_WORDS} words"),
                      _mp.Patch(color="#8A9098", label="held: non-narrative item")],
             fontsize=7.5, frameon=False, loc="lower right")
for q in pooled_qs:
    m = origin_pool == q
    ax[1].scatter(pca2[m, 0], pca2[m, 1], s=9, alpha=0.50, color=Q_COLORS[q],
                  label=q, edgecolors="none")
ax[1].legend(fontsize=8, frameon=False, markerscale=2.2, loc="upper right")
ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_xlabel(f"PC1 ({v0:.0%} var)"); ax[1].set_ylabel(f"PC2 ({v1:.0%} var)")
ax[1].set_title("B  Pooled space (PCA-2D): the narrative questions overlap, so pooling is justified",
                fontsize=10.5, fontweight="bold", loc="left", color=NT)
for a in ax:
    a.set_facecolor("white")
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
ax[0].grid(True, axis="x", color="#F0F0F0"); ax[0].set_axisbelow(True)
fig.suptitle(f"Thematic pool: {total} responses from {len(pooled_qs)} narrative questions "
             f"({CFG.embed_dim}-d)   ·   {len(X_held)} held out for post-hoc assignment",
             fontsize=12.5, fontweight="bold", color=NT, y=1.02)
fig.text(0.5, -0.04, "Held-out rows are not discarded: they keep their embeddings and are assigned a theme "
         "post-hoc in NB2 with an explicit confidence margin; the One-Word item additionally gets its own "
         "lexical analysis in NB3.", ha="center", fontsize=8, style="italic", color="#888")
plt.tight_layout(); save_fig(fig, "chart_stage18_pooling_preview"); plt.close(fig)
print("=" * 78)


## Notebook 1 - Manifest & artifact diagnostic
Fingerprints, version, timestamp and the artifact inventory go into `embed_manifest.json`; the diagnostic confirms the hand-off set is complete.

In [ ]:
# @title
# ==============================================================================
# NOTEBOOK 1  -  WRITE MANIFEST + ARTIFACT DIAGNOSTIC
# Records the dataset + config fingerprints, the notebook version, a timestamp,
# and every artifact written this session into embed_manifest.json (local + mirror),
# then confirms the seven core hand-off artifacts are present. Notebook 2 reads
# this manifest to verify the Control Panel has not changed underneath it.
# ==============================================================================
import numpy as np

# Re-acquire the hand-off objects (works whether or not the cells above are in memory).
X_pool      = np.asarray(pload("pooled_embeddings"), dtype=np.float32)
meta_pool   = pload("pooled_meta")
origin_pool = np.asarray(pload("pooled_origin"))
n_rows      = len(meta_pool)

_policy = pload("pooled_pool_policy") or {}
fingerprints = dict(
    data=CFG.fingerprint("data", 0, extra=CFG.data_url),
    embed=CFG.fingerprint("embed", n_rows, extra=data_sig(X_pool)),
    dataset_embeddings=data_sig(X_pool),
    # pooling policy travels with the manifest so NB2/NB3 and the methods section
    # can state exactly what entered the thematic space and why.
    pool_exclude_questions=_policy.get("exclude_questions"),
    pool_min_words=_policy.get("min_pool_words"),
    n_embedded=_policy.get("n_embedded"),
    n_pooled=_policy.get("n_core"),
    n_held_out=_policy.get("n_held"),
)

REQUIRED_NB1 = ["df_cleaned", "df_final", "embeddings_dict", "metadata_dict",
                "pooled_embeddings", "pooled_origin", "pooled_meta"]

write_manifest("embed_manifest.json", fingerprints, n_rows=n_rows)
missing = artifact_diagnostic(REQUIRED_NB1,
                              optional=["embedding_quality", "pooled_held_embeddings",
                                        "pooled_held_origin", "pooled_held_meta",
                                        "pooled_pool_policy"])

print()
if missing:
    print(f"  WARNING: {len(missing)} required artifact(s) missing — re-run the stage(s) that produce them.")
else:
    print("  Notebook 1 complete. All hand-off artifacts are saved.")
    if _policy:
        print(f"  thematic pool : {_policy.get('n_core')} rows "
              f"(of {_policy.get('n_embedded')} embedded)")
        print(f"  held out      : {_policy.get('n_held')} rows -> pooled_held_* "
              f"(NB2 assigns post-hoc; nothing discarded)")
        print(f"  policy        : exclude={_policy.get('exclude_questions')}, "
              f"min_words={_policy.get('min_pool_words')}")
    print(f"  Next: open Notebook 2. It will load these from {POOLED_DIR}")
    if MIRROR_POOLED_DIR:
        print(f"        (mirror copy at {MIRROR_POOLED_DIR}; big files stay local-only).")
